In [67]:
# ═══════════════════════════════════════════════════════════════
# CUSTOMER CHURN PREDICTION — BANK DATASET
# Goal: Predict which customers will leave the bank (Exited=1)
# Dataset: 10,000 customers, 14 features, binary target
# ═══════════════════════════════════════════════════════════════

# Import core libraries for data manipulation, visualization, and numerical ops
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


In [68]:
# Load raw dataset (10K rows × 14 columns) and preview first 5 rows
# Columns: RowNumber, CustomerId, Surname (drop later), CreditScore, Geography,
# Gender, Age, Tenure, Balance, NumOfProducts, HasCrCard, IsActiveMember,
# EstimatedSalary, Exited (TARGET — 1 = churned, 0 = stayed)
df = pd.read_csv('data\\Churn_Modelling.csv')
df.head()


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [69]:
# Check data types, null counts, and memory usage
# Key observations: No nulls, 3 object columns (Surname, Geography, Gender)
# need encoding, rest are numeric
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [70]:
# Descriptive statistics for all numeric columns
# Key insight: Balance has high std (62K) and min=0 (36% zero-balance customers)
# Age range 18-92, mean ~39 — Age will be a strong predictor
# Exited mean=0.2037 → ~80/20 class imbalance (important for later modeling)
df.describe()


,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.00000,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,5000.50000,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,2886.89568,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.00000,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2500.75000,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,5000.50000,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,7500.25000,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,10000.00000,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


In [71]:
# Check target class distribution
# Result: 7963 stayed (0) vs 2037 churned (1) → 3.9:1 imbalance ratio
# This imbalance means accuracy is misleading — models can get 80% by always
# predicting 'stayed'. Must use F1/Recall as primary metric + SMOTE/class_weight
df.Exited.value_counts()


Exited
0    7963
1    2037
Name: count, dtype: int64

In [72]:
# Encode Gender: Female→0, Male→1 using LabelEncoder
# Binary categories with no ordinal meaning — LabelEncoder is appropriate here
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])


In [73]:
# One-hot encode Geography (France/Germany/Spain) into binary columns
# Note: OHE output is a sparse array stuffed into one column here — this is
# technically a bug (should use pd.get_dummies). Fixed in the final pipeline.
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder()
df['Geography'] = ohe.fit_transform(df['Geography'].values.reshape(-1,1)).toarray()


In [74]:
# Extract feature matrix X (columns 3-12) and target vector y (column 13)
# iloc slicing selects: CreditScore, Geography(encoded), Gender, Age, Tenure,
# Balance, NumOfProducts, HasCrCard, IsActiveMember, EstimatedSalary
X = df.iloc[:, 3:13].values
y = df.iloc[:, 13].values


In [75]:
# ═══════════════════════════════════════════════════════════════
# INITIAL EXPERIMENT: Logistic Regression with manual setup
# Purpose: Establish a baseline before the automated pipeline
# ═══════════════════════════════════════════════════════════════

# Steps: Train-test split → StandardScaler → LogReg with class_weight='balanced'
# → 5-fold CV → Confusion matrix + classification report
# Result: ~70% accuracy, 0.72 recall on churners — decent baseline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=0
)

# Scaling
sc = StandardScaler()

X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# Logistic Regression
classifier = LogisticRegression(
    C=0.01,
    max_iter=1000,
    solver='liblinear',
    class_weight='balanced',
    random_state=0
)

# 5-Fold Cross Validation on Training Data
cv_scores = cross_val_score(
    classifier,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("Cross Validation Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())
print("Std CV Accuracy:", cv_scores.std())

# Train on Full Training Set
classifier.fit(X_train, y_train)

# Predictions on Test Set
y_pred = classifier.predict(X_test)

# Evaluation
cm = confusion_matrix(y_test, y_pred)
print(cm)

print("Test Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Cross Validation Scores: [0.7025   0.69875  0.70125  0.704375 0.69625 ]
Mean CV Accuracy: 0.7006249999999999
Std CV Accuracy: 0.002850438562747833
[[1098  497]
 [ 114  291]]
Test Accuracy: 0.6945
              precision    recall  f1-score   support

           0       0.91      0.69      0.78      1595
           1       0.37      0.72      0.49       405

    accuracy                           0.69      2000
   macro avg       0.64      0.70      0.64      2000
weighted avg       0.80      0.69      0.72      2000



In [87]:
# ═══════════════════════════════════════════════════════════════
# GridSearchCV on LogReg: Tune C, penalty, solver, class_weight
# ═══════════════════════════════════════════════════════════════
# WARNING: C grid has 10,000 values × 2 penalties × 2 class_weights = 40,000 combos
# × 5 folds = 200,000 fits — this is why it hits KeyboardInterrupt!
# Fix: Use RandomizedSearchCV with n_iter=30 (as done in final pipeline)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000))
])

param_grid = {
    'logreg__C': np.linspace(0.001, 100, 10000),
    'logreg__penalty': ['l1','l2'],
    'logreg__solver': ['liblinear'],
    'logreg__class_weight': [None,'balanced']
}

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train, y_train)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

KeyboardInterrupt: 

In [78]:
# Print the best hyperparameter combination found by GridSearchCV
# and evaluate the winning model on the held-out test set
print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)


Best Parameters: {'logreg__C': 0.01, 'logreg__class_weight': 'balanced', 'logreg__penalty': 'l1', 'logreg__solver': 'liblinear'}
Best Score: 0.4878198397342727


In [85]:
# ═══════════════════════════════════════════════════════════════
# THRESHOLD TUNING (Early Experiment)
# ═══════════════════════════════════════════════════════════════
# Default threshold is 0.5, but with imbalanced data the optimal
# cutoff is usually lower (0.35-0.45). This loop finds the threshold
# that maximizes F1 score on the test set.
y_prob = best_model.predict_proba(X_test)[:,1]
from sklearn.metrics import f1_score

for t in [0.3,0.4,0.5,0.6,0.7]:
    y_pred = (y_prob >= t).astype(int)
    print(t, f1_score(y_test,y_pred))

0.3 0.3896761133603239
0.4 0.44139650872817954
0.5 0.4897959183673469
0.6 0.5027203482045702
0.7 0.4198250728862974


In [86]:
# ROC-AUC: Area Under the Receiver Operating Characteristic curve
# Unlike F1, ROC-AUC is threshold-independent — it measures how well
# the model separates classes across ALL possible thresholds
# AUC=1.0 is perfect, AUC=0.5 is random guessing
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test,y_prob)


0.7722140949727158

In [88]:
# ═══════════════════════════════════════════════════════════════
# MODEL COMPARISON — Round 1 (Default settings, no tuning)
# ═══════════════════════════════════════════════════════════════
# Reusable evaluate_model() function: fits, predicts, and stores
# Accuracy, F1, and ROC-AUC for each classifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import pandas as pd

results = []


In [89]:
# Logistic Regression baseline — linear decision boundary
# Good for linearly separable problems, fast, interpretable
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

results.append([
    "Logistic",
    accuracy_score(y_test,y_pred),
    f1_score(y_test,y_pred),
    roc_auc_score(y_test,y_prob)
])

In [90]:
# Decision Tree baseline — no max_depth limit → will overfit training data
# But useful as a variance reference point
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=0)

model.fit(X_train,y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

results.append([
    "Decision Tree",
    accuracy_score(y_test,y_pred),
    f1_score(y_test,y_pred),
    roc_auc_score(y_test,y_prob)
])

In [97]:
# Preview accumulated model results so far
results


[['Logistic', 0.811, 0.3102189781021898, 0.7640419520879291],
 ['Decision Tree', 0.7805, 0.4889406286379511, 0.682770231046093],
 ['Random Forest', 0.8625, 0.5925925925925926, 0.8617972831765934],
 ['Gradient Boosting', 0.8635, 0.5869894099848714, 0.8672255118232131],
 ['XGBoost', 0.8515, 0.5892116182572614, 0.8500623089128837]]

In [92]:
# Random Forest (200 trees) — bagging ensemble that averages many trees
# Reduces variance compared to single Decision Tree
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=0
)

model.fit(X_train,y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

results.append([
    "Random Forest",
    accuracy_score(y_test,y_pred),
    f1_score(y_test,y_pred),
    roc_auc_score(y_test,y_prob)
])

In [94]:
# Gradient Boosting — builds trees sequentially, each correcting prior errors
# Typically the strongest classical ML model on tabular data
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier()

model.fit(X_train,y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

results.append([
    "Gradient Boosting",
    accuracy_score(y_test,y_pred),
    f1_score(y_test,y_pred),
    roc_auc_score(y_test,y_prob)
])

In [106]:
# XGBoost — regularized gradient boosting with L1/L2 penalties
# logloss eval metric for binary classification
# Often outperforms sklearn's GBM due to better regularization
from xgboost import XGBClassifier

model = XGBClassifier(
    eval_metric='logloss',
    random_state=0
)

model.fit(X_train,y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

results.append([
    "XGBoost",
    accuracy_score(y_test,y_pred),
    f1_score(y_test,y_pred),
    roc_auc_score(y_test,y_prob)
])

In [107]:
# Compile Round 1 results into a DataFrame sorted by ROC-AUC
# This gives a quick leaderboard of all default models
results_df = pd.DataFrame(
    results,
    columns=["Model","Accuracy","F1","ROC_AUC"]
)

print(results_df.sort_values(
    "ROC_AUC",
    ascending=False
))

               Model  Accuracy        F1   ROC_AUC
3  Gradient Boosting    0.8635  0.586989  0.867226
2      Random Forest    0.8625  0.592593  0.861797
4            XGBoost    0.8515  0.589212  0.850062
5            XGBoost    0.8515  0.589212  0.850062
6            XGBoost    0.8515  0.589212  0.850062
0           Logistic    0.8110  0.310219  0.764042
1      Decision Tree    0.7805  0.488941  0.682770


In [109]:
# ═══════════════════════════════════════════════════════════════
# MODEL COMPARISON — Round 2 (Tuned with GridSearchCV)
# ═══════════════════════════════════════════════════════════════
# Reusable helper with GridSearchCV built in
# Fits default → stores metrics → tunes with grid → stores tuned metrics
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import pandas as pd

results_1 = []

def evaluate_model(name, model, X_train, X_test, y_train, y_test):

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    results_1.append([
        name,
        accuracy_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_prob),
        "Baseline"
    ])

In [110]:
# KNN baseline — distance-based classifier, sensitive to feature scaling
# (StandardScaler already applied above)
from sklearn.neighbors import KNeighborsClassifier

evaluate_model(
    "KNN",
    KNeighborsClassifier(),
    X_train,
    X_test,
    y_train,
    y_test
)

In [111]:
# Logistic Regression baseline added to the second results list
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=1000,
    random_state=0
)

evaluate_model(
    "Logistic",
    log_model,
    X_train,
    X_test,
    y_train,
    y_test
)

In [112]:
# Tune Logistic Regression: search over regularization (C), penalty type,
# and class_weight. Scoring on ROC-AUC.
param_grid = {
    'C':[0.001,0.01,0.1,1,10,100,1000],
    'penalty':['l1','l2'],
    'class_weight':[None,'balanced'],
    'solver':['liblinear']
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:,1]

results.append([
    "Logistic Tuned",
    accuracy_score(y_test,y_pred),
    f1_score(y_test,y_pred),
    roc_auc_score(y_test,y_prob),
    str(grid.best_params_)
])

c:\Users\Admin\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Admin\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [113]:
# KNN baseline added to results_1 for side-by-side comparison with tuned KNN below
from sklearn.neighbors import KNeighborsClassifier

evaluate_model(
    "KNN",
    KNeighborsClassifier(),
    X_train,
    X_test,
    y_train,
    y_test
)

In [114]:
# Tune KNN: search over k (neighbor count), weighting scheme, and distance metric
param_grid = {
    'n_neighbors':[3,5,7,9,11,15,21],
    'weights':['uniform','distance'],
    'metric':['euclidean','manhattan']
}

grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

grid.fit(X_train,y_train)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:,1]

results.append([
    "KNN Tuned",
    accuracy_score(y_test,y_pred),
    f1_score(y_test,y_pred),
    roc_auc_score(y_test,y_prob),
    str(grid.best_params_)
])

In [115]:
# SVM baseline — probability=True enables predict_proba for ROC-AUC
# SVM with RBF kernel maps data into higher dimensions
from sklearn.svm import SVC

evaluate_model(
    "SVM",
    SVC(probability=True),
    X_train,
    X_test,
    y_train,
    y_test
)

In [116]:
# Tune SVM: search over regularization (C), kernel width (gamma), kernel type
# Note: SVM is O(n²) so this is slow on 8000+ samples
param_grid = {
    'C':[0.01,0.1,1,10,100],
    'gamma':[0.001,0.01,0.1,1],
    'kernel':['rbf']
}

grid = GridSearchCV(
    SVC(probability=True),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

grid.fit(X_train,y_train)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:,1]

results.append([
    "SVM Tuned",
    accuracy_score(y_test,y_pred),
    f1_score(y_test,y_pred),
    roc_auc_score(y_test,y_prob),
    str(grid.best_params_)
])

In [117]:
# Compile Round 2 results (default + tuned) sorted by ROC-AUC
results_df = pd.DataFrame(
    results_1,
    columns=[
        "Model",
        "Accuracy",
        "F1",
        "ROC_AUC",
        "Best_Params"
    ]
)

results_df = results_df.sort_values(
    by="ROC_AUC",
    ascending=False
)

print(results_df)

      Model  Accuracy        F1   ROC_AUC Best_Params
3       SVM     0.862  0.543046  0.832733    Baseline
0       KNN     0.830  0.483283  0.781289    Baseline
2       KNN     0.830  0.483283  0.781289    Baseline
1  Logistic     0.811  0.310219  0.764042    Baseline


In [9]:
# ═══════════════════════════════════════════════════════════════
# ATTEMPT 3: Full pipeline WITHOUT class balancing (SMOTE/class_weight)
# Result: Recall stuck at 16-46% — models predict majority class
# Conclusion: Class imbalance is the bottleneck, not model choice
# ═══════════════════════════════════════════════════════════════
"""
=============================================================================
END-TO-END CLASSIFIER BENCHMARK PIPELINE
- 7 Classifiers: Logistic, KNN, SVM, Decision Tree, Random Forest,
                  Gradient Boosting, XGBoost
- Each run in DEFAULT and TUNED (RandomizedSearchCV) mode
- 5-Fold Cross-Validation on all
- Single reproducible DataFrame output
=============================================================================
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from time import time

# sklearn utilities
from sklearn.model_selection import (
    train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# ─── REPRODUCIBILITY ───
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ─── LOAD DATA ───
# >>>>>> OPTION 1: Your own CSV <<<<<<
df = pd.read_csv(r"C:\Users\Admin\Desktop\Notes\ML\churn\data\Churn_Modelling.csv")
# TARGET_COL = "your_target_column_name"   # <-- change this
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])

from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder()
df['Geography'] = ohe.fit_transform(df['Geography'].values.reshape(-1,1)).toarray()

X = df.iloc[:, 3:13].values
y = df.iloc[:, 13].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Samples: {X.shape[0]} | Features: {X.shape[1]}")
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
print("=" * 80)

# ─── DEFINE CLASSIFIERS + HYPERPARAMETER GRIDS ───
CLASSIFIERS = {
    "Logistic Regression": {
        "model": LogisticRegression(
            max_iter=5000, random_state=RANDOM_STATE
        ),
        "params": {
            "classifier__C": np.logspace(-3, 3, 20),
            "classifier__penalty": ["l1", "l2"],
            "classifier__solver": ["liblinear", "saga"],
        },
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            "classifier__n_neighbors": list(range(3, 31, 2)),
            "classifier__weights": ["uniform", "distance"],
            "classifier__metric": ["euclidean", "manhattan", "minkowski"],
            "classifier__p": [1, 2, 3],
        },
    },

    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "params": {
            "classifier__max_depth": [3, 5, 7, 10, 15, 20, None],
            "classifier__min_samples_split": [2, 5, 10, 20],
            "classifier__min_samples_leaf": [1, 2, 5, 10],
            "classifier__criterion": ["gini", "entropy"],
            "classifier__max_features": ["sqrt", "log2", None],
        },
    },
    "Random Forest": {
        "model": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__max_depth": [5, 10, 20],
            "classifier__min_samples_split": [2, 5],
            "classifier__max_features": ["sqrt", "log2"],
        },
    },
    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__learning_rate": [0.05, 0.1, 0.2],
            "classifier__max_depth": [3, 5, 7],
            "classifier__subsample": [0.8, 1.0],
        },
    },
    "XGBoost": {
        "model": XGBClassifier(
            eval_metric="logloss",
            use_label_encoder=False,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__learning_rate": [0.05, 0.1, 0.2],
            "classifier__max_depth": [3, 5, 7],
            "classifier__subsample": [0.8, 1.0],
            "classifier__colsample_bytree": [0.8, 1.0],
        },
    },
}


# ─── EVALUATION FUNCTION ───
def evaluate_model(pipeline, X_tr, X_te, y_tr, y_te):
    """Fit on train, score on test, return metrics dict."""
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    y_proba = (
        pipeline.predict_proba(X_te)[:, 1]
        if hasattr(pipeline, "predict_proba")
        else None
    )
    metrics = {
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred, zero_division=0),
        "F1": f1_score(y_te, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_te, y_proba) if y_proba is not None else np.nan,
    }
    return metrics


# ─── MAIN PIPELINE ───
results = []

for name, config in CLASSIFIERS.items():
    print(f"\n{'─' * 60}")
    print(f"  {name}")
    print(f"{'─' * 60}")

    # ── 1. DEFAULT (no tuning) ──
    t0 = time()
    pipe_default = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", config["model"]),
    ])

    # 5-fold CV scores (on training set)
    cv_scores = cross_val_score(
        pipe_default, X_train, y_train, cv=CV, scoring="accuracy"
    )

    # Train-test metrics
    metrics = evaluate_model(pipe_default, X_train, X_test, y_train, y_test)
    elapsed = time() - t0

    results.append({
        "Model": name,
        "Mode": "Default",
        "CV_Mean_Accuracy": round(cv_scores.mean(), 4),
        "CV_Std": round(cv_scores.std(), 4),
        "Test_Accuracy": round(metrics["Accuracy"], 4),
        "Test_Precision": round(metrics["Precision"], 4),
        "Test_Recall": round(metrics["Recall"], 4),
        "Test_F1": round(metrics["F1"], 4),
        "Test_ROC_AUC": round(metrics["ROC-AUC"], 4),
        "Best_Params": "default",
        "Time_sec": round(elapsed, 2),
    })

    print(f"  [Default]  CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}  "
          f"Test_Acc={metrics['Accuracy']:.4f}  ({elapsed:.1f}s)")

    # ── 2. TUNED (RandomizedSearchCV) ──
    t0 = time()
    pipe_tuned = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", config["model"]),
    ])

    search = RandomizedSearchCV(
        estimator=pipe_tuned,
        param_distributions=config["params"],
        n_iter=10,
        cv=CV,
        scoring="accuracy",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        return_train_score=False,
    )
    search.fit(X_train, y_train)

    # Evaluate best estimator on test set
    best_pipe = search.best_estimator_
    y_pred = best_pipe.predict(X_test)
    y_proba = (
        best_pipe.predict_proba(X_test)[:, 1]
        if hasattr(best_pipe, "predict_proba")
        else None
    )

    metrics_tuned = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan,
    }
    elapsed = time() - t0

    # Clean param names for display (strip 'classifier__' prefix)
    best_params = {
        k.replace("classifier__", ""): v
        for k, v in search.best_params_.items()
    }

    results.append({
        "Model": name,
        "Mode": "Tuned (RandomizedSearchCV)",
        "CV_Mean_Accuracy": round(search.best_score_, 4),
        "CV_Std": round(
            search.cv_results_["std_test_score"][search.best_index_], 4
        ),
        "Test_Accuracy": round(metrics_tuned["Accuracy"], 4),
        "Test_Precision": round(metrics_tuned["Precision"], 4),
        "Test_Recall": round(metrics_tuned["Recall"], 4),
        "Test_F1": round(metrics_tuned["F1"], 4),
        "Test_ROC_AUC": round(metrics_tuned["ROC-AUC"], 4),
        "Best_Params": str(best_params),
        "Time_sec": round(elapsed, 2),
    })

    print(f"  [Tuned]    CV={search.best_score_:.4f}  "
          f"Test_Acc={metrics_tuned['Accuracy']:.4f}  ({elapsed:.1f}s)")
    print(f"  Best → {best_params}")


# ─── BUILD FINAL DATAFRAME ───
df_results = pd.DataFrame(results)

# Sort: best test accuracy at top
df_results = df_results.sort_values("Test_Accuracy", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("  FINAL RESULTS — ALL CLASSIFIERS (Default + Tuned)")
print("=" * 80)

# Display-friendly print
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
print(df_results.to_string(index=False))

# ─── SAVE ───
output_path = "/mnt/user-data/outputs/classifier_comparison_results.csv"
df_results.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")

# ─── QUICK SUMMARY ───
print("\n" + "=" * 80)
print("  TOP 3 MODELS BY TEST ACCURACY")
print("=" * 80)
for i, row in df_results.head(3).iterrows():
    print(f"  {i+1}. {row['Model']} ({row['Mode']}) — "
          f"Acc: {row['Test_Accuracy']}  F1: {row['Test_F1']}  "
          f"AUC: {row['Test_ROC_AUC']}")

Samples: 10000 | Features: 10
Train: 8000 | Test: 2000
Class distribution: {np.int64(0): np.int64(7963), np.int64(1): np.int64(2037)}

────────────────────────────────────────────────────────────
  Logistic Regression
────────────────────────────────────────────────────────────
  [Default]  CV=0.8101±0.0035  Test_Acc=0.8090  (0.1s)
  [Tuned]    CV=0.8101  Test_Acc=0.8090  (9.5s)
  Best → {'solver': 'liblinear', 'penalty': 'l2', 'C': np.float64(233.57214690901213)}

────────────────────────────────────────────────────────────
  KNN
────────────────────────────────────────────────────────────
  [Default]  CV=0.8226±0.0117  Test_Acc=0.8310  (0.5s)
  [Tuned]    CV=0.8349  Test_Acc=0.8350  (2.0s)
  Best → {'weights': 'distance', 'p': 1, 'n_neighbors': 13, 'metric': 'manhattan'}

────────────────────────────────────────────────────────────
  Decision Tree
────────────────────────────────────────────────────────────
  [Default]  CV=0.7808±0.0060  Test_Acc=0.7780  (0.2s)
  [Tuned]    CV=0.8524

OSError: Cannot save file into a non-existent directory: '\mnt\user-data\outputs'

In [11]:
# ═══════════════════════════════════════════════════════════════
# ATTEMPT 4: Pipeline WITH class_weight='balanced' + scoring='f1'
# Added class_weight to LR, SVM, DT, RF; scale_pos_weight to XGBoost
# Changed scoring from 'accuracy' to 'f1'
# Result: Recall improved to 46-78% but F1 still capped at ~0.58
# ═══════════════════════════════════════════════════════════════
"""
=============================================================================
END-TO-END CLASSIFIER BENCHMARK PIPELINE
- 7 Classifiers: Logistic, KNN, SVM, Decision Tree, Random Forest,
                  Gradient Boosting, XGBoost
- Each run in DEFAULT and TUNED (RandomizedSearchCV) mode
- 5-Fold Cross-Validation on all
- Single reproducible DataFrame output
=============================================================================
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from time import time
!pip install imbalanced-learn

# sklearn utilities
from sklearn.model_selection import (
    train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# ─── REPRODUCIBILITY ───
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ─── LOAD DATA ───
# >>>>>> OPTION 1: Your own CSV <<<<<<
df = pd.read_csv(r"C:\Users\Admin\Desktop\Notes\ML\churn\data\Churn_Modelling.csv")
# TARGET_COL = "your_target_column_name"   # <-- change this
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])

from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder()
df['Geography'] = ohe.fit_transform(df['Geography'].values.reshape(-1,1)).toarray()

X = df.iloc[:, 3:13].values
y = df.iloc[:, 13].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Samples: {X.shape[0]} | Features: {X.shape[1]}")
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
print("=" * 80)

# Compute imbalance ratio for XGBoost
neg_count = sum(1 for val in y_train if val == 0)
pos_count = sum(1 for val in y_train if val == 1)
scale_pos = neg_count / pos_count if pos_count > 0 else 1.0
print(f"Imbalance ratio (neg/pos): {scale_pos:.2f}")
print("=" * 80)

# ─── DEFINE CLASSIFIERS + HYPERPARAMETER GRIDS ───
# FIX: class_weight='balanced' on all models that support it
#      scale_pos_weight on XGBoost
#      Scoring changed to 'f1' instead of 'accuracy'
CLASSIFIERS = {
    "Logistic Regression": {
        "model": LogisticRegression(
            max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "params": {
            "classifier__C": np.logspace(-3, 3, 20),
            "classifier__penalty": ["l1", "l2"],
            "classifier__solver": ["liblinear", "saga"],
        },
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            "classifier__n_neighbors": list(range(3, 31, 2)),
            "classifier__weights": ["uniform", "distance"],
            "classifier__metric": ["euclidean", "manhattan"],
            "classifier__p": [1, 2],
        },
    },
    
    "Decision Tree": {
        "model": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "params": {
            "classifier__max_depth": [3, 5, 7, 10, 15, 20, None],
            "classifier__min_samples_split": [2, 5, 10, 20],
            "classifier__min_samples_leaf": [1, 2, 5, 10],
            "classifier__criterion": ["gini", "entropy"],
            "classifier__max_features": ["sqrt", "log2", None],
        },
    },
    "Random Forest": {
        "model": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__max_depth": [5, 10, 20],
            "classifier__min_samples_split": [2, 5],
            "classifier__max_features": ["sqrt", "log2"],
        },
    },
    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__learning_rate": [0.05, 0.1, 0.2],
            "classifier__max_depth": [3, 5, 7],
            "classifier__subsample": [0.8, 1.0],
        },
    },
    "XGBoost": {
        "model": XGBClassifier(
            eval_metric="logloss",
            use_label_encoder=False,
            scale_pos_weight=scale_pos,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__learning_rate": [0.05, 0.1, 0.2],
            "classifier__max_depth": [3, 5, 7],
            "classifier__subsample": [0.8, 1.0],
            "classifier__colsample_bytree": [0.8, 1.0],
        },
    },
}


# ─── EVALUATION FUNCTION ───
def evaluate_model(pipeline, X_tr, X_te, y_tr, y_te):
    """Fit on train, score on test, return metrics dict."""
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    y_proba = (
        pipeline.predict_proba(X_te)[:, 1]
        if hasattr(pipeline, "predict_proba")
        else None
    )
    metrics = {
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred, zero_division=0),
        "F1": f1_score(y_te, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_te, y_proba) if y_proba is not None else np.nan,
    }
    return metrics


# ─── MAIN PIPELINE ───
results = []

for name, config in CLASSIFIERS.items():
    print(f"\n{'─' * 60}")
    print(f"  {name}")
    print(f"{'─' * 60}")

    # ── 1. DEFAULT (no tuning) ──
    t0 = time()
    pipe_default = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", config["model"]),
    ])

    # 5-fold CV scores (on training set)
    cv_scores = cross_val_score(
        pipe_default, X_train, y_train, cv=CV, scoring="f1"
    )

    # Train-test metrics
    metrics = evaluate_model(pipe_default, X_train, X_test, y_train, y_test)
    elapsed = time() - t0

    results.append({
        "Model": name,
        "Mode": "Default",
        "CV_Mean_F1": round(cv_scores.mean(), 4),
        "CV_Std": round(cv_scores.std(), 4),
        "Test_Accuracy": round(metrics["Accuracy"], 4),
        "Test_Precision": round(metrics["Precision"], 4),
        "Test_Recall": round(metrics["Recall"], 4),
        "Test_F1": round(metrics["F1"], 4),
        "Test_ROC_AUC": round(metrics["ROC-AUC"], 4),
        "Best_Params": "default",
        "Time_sec": round(elapsed, 2),
    })

    print(f"  [Default]  CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}  "
          f"Test_Acc={metrics['Accuracy']:.4f}  ({elapsed:.1f}s)")

    # ── 2. TUNED (RandomizedSearchCV) ──
    t0 = time()
    pipe_tuned = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", config["model"]),
    ])

    search = RandomizedSearchCV(
        estimator=pipe_tuned,
        param_distributions=config["params"],
        n_iter=10,
        cv=CV,
        scoring="f1",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        return_train_score=False,
    )
    search.fit(X_train, y_train)

    # Evaluate best estimator on test set
    best_pipe = search.best_estimator_
    y_pred = best_pipe.predict(X_test)
    y_proba = (
        best_pipe.predict_proba(X_test)[:, 1]
        if hasattr(best_pipe, "predict_proba")
        else None
    )

    metrics_tuned = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan,
    }
    elapsed = time() - t0

    # Clean param names for display (strip 'classifier__' prefix)
    best_params = {
        k.replace("classifier__", ""): v
        for k, v in search.best_params_.items()
    }

    results.append({
        "Model": name,
        "Mode": "Tuned (RandomizedSearchCV)",
        "CV_Mean_F1": round(search.best_score_, 4),
        "CV_Std": round(
            search.cv_results_["std_test_score"][search.best_index_], 4
        ),
        "Test_Accuracy": round(metrics_tuned["Accuracy"], 4),
        "Test_Precision": round(metrics_tuned["Precision"], 4),
        "Test_Recall": round(metrics_tuned["Recall"], 4),
        "Test_F1": round(metrics_tuned["F1"], 4),
        "Test_ROC_AUC": round(metrics_tuned["ROC-AUC"], 4),
        "Best_Params": str(best_params),
        "Time_sec": round(elapsed, 2),
    })

    print(f"  [Tuned]    CV={search.best_score_:.4f}  "
          f"Test_Acc={metrics_tuned['Accuracy']:.4f}  ({elapsed:.1f}s)")
    print(f"  Best → {best_params}")


# ─── BUILD FINAL DATAFRAME ───
df_results = pd.DataFrame(results)

# Sort: best test accuracy at top
df_results = df_results.sort_values("Test_F1", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("  FINAL RESULTS — ALL CLASSIFIERS (Default + Tuned)")
print("=" * 80)

# Display-friendly print
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
print(df_results.to_string(index=False))



# ─── QUICK SUMMARY ───
print("\n" + "=" * 80)
print("  TOP 3 MODELS BY TEST ACCURACY")
print("=" * 80)
for i, row in df_results.head(3).iterrows():
    print(f"  {i+1}. {row['Model']} ({row['Mode']}) — "
          f"Acc: {row['Test_Accuracy']}  F1: {row['Test_F1']}  "
          f"AUC: {row['Test_ROC_AUC']}")

Samples: 10000 | Features: 10
Train: 8000 | Test: 2000
Class distribution: {np.int64(0): np.int64(7963), np.int64(1): np.int64(2037)}
Imbalance ratio (neg/pos): 3.91

────────────────────────────────────────────────────────────
  Logistic Regression
────────────────────────────────────────────────────────────
  [Default]  CV=0.4796±0.0139  Test_Acc=0.7085  (0.8s)
  [Tuned]    CV=0.4819  Test_Acc=0.6970  (37.9s)
  Best → {'solver': 'liblinear', 'penalty': 'l1', 'C': np.float64(0.008858667904100823)}

────────────────────────────────────────────────────────────
  KNN
────────────────────────────────────────────────────────────
  [Default]  CV=0.4395±0.0450  Test_Acc=0.8310  (3.1s)
  [Tuned]    CV=0.4400  Test_Acc=0.8290  (4.6s)
  Best → {'weights': 'distance', 'p': 2, 'n_neighbors': 7, 'metric': 'euclidean'}

────────────────────────────────────────────────────────────
  Decision Tree
────────────────────────────────────────────────────────────
  [Default]  CV=0.4690±0.0310  Test_Acc=0.7

In [13]:
# ═══════════════════════════════════════════════════════════════
# ATTEMPT 5: Added SMOTE (Synthetic Minority Over-sampling)
# Uses imblearn.pipeline.Pipeline (NOT sklearn) so SMOTE only
# applies to training folds, never leaks into test data
# Triple defense: SMOTE + class_weight + F1 scoring
# Result: F1 ~0.58-0.62 — improvement but still below deployment threshold
# ═══════════════════════════════════════════════════════════════
"""
=============================================================================
END-TO-END CLASSIFIER BENCHMARK PIPELINE (WITH SMOTE)
- 7 Classifiers: Logistic, KNN, SVM, Decision Tree, Random Forest,
                  Gradient Boosting, XGBoost
- Each run in DEFAULT and TUNED (RandomizedSearchCV) mode
- 5-Fold Cross-Validation on all
- SMOTE for class balancing
- Single reproducible DataFrame output
=============================================================================
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from time import time

# pip install imbalanced-learn xgboost   <-- run ONCE in terminal before running

# sklearn utilities
from sklearn.model_selection import (
    train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# SMOTE + imblearn Pipeline (applies SMOTE only on train folds, never test)
# Fix sklearn / imbalanced-learn version mismatch
%pip install -q -U imbalanced-learn scikit-learn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline   # NOT sklearn.pipeline.Pipeline

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# ─── REPRODUCIBILITY ───
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ─── LOAD DATA ───
df = pd.read_csv(r"C:\Users\Admin\Desktop\Notes\ML\churn\data\Churn_Modelling.csv")

# Encode Gender
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])

# One-Hot Encode Geography (FIXED — pd.get_dummies, not OHE on single column)
df = pd.get_dummies(df, columns=['Geography'], drop_first=True)

# Features and target
X = df.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Exited'])
y = df['Exited']

# DO NOT scale here — scaling happens INSIDE the pipeline (prevents data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Samples: {X.shape[0]} | Features: {X.shape[1]}")
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
class_counts = dict(zip(*np.unique(y, return_counts=True)))
print(f"Class distribution: {class_counts}")

# Compute imbalance ratio for XGBoost
neg_count = sum(1 for val in y_train if val == 0)
pos_count = sum(1 for val in y_train if val == 1)
scale_pos = neg_count / pos_count if pos_count > 0 else 1.0
print(f"Imbalance ratio (neg/pos): {scale_pos:.2f}")
print("=" * 80)

# ─── DEFINE CLASSIFIERS + HYPERPARAMETER GRIDS ───
CLASSIFIERS = {
    "Logistic Regression": {
        "model": LogisticRegression(
            max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "params": {
            "classifier__C": np.logspace(-3, 3, 20),
            "classifier__penalty": ["l1", "l2"],
            "classifier__solver": ["liblinear", "saga"],
        },
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            "classifier__n_neighbors": list(range(3, 31, 2)),
            "classifier__weights": ["uniform", "distance"],
            "classifier__metric": ["euclidean", "manhattan"],
            "classifier__p": [1, 2],
        },
    },
    "SVM": {
        "model": CalibratedClassifierCV(
            LinearSVC(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE), cv=3
        ),
        "params": {
            "classifier__estimator__C": np.logspace(-2, 2, 8),
        },
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "params": {
            "classifier__max_depth": [3, 5, 7, 10, 15, 20, None],
            "classifier__min_samples_split": [2, 5, 10, 20],
            "classifier__min_samples_leaf": [1, 2, 5, 10],
            "classifier__criterion": ["gini", "entropy"],
            "classifier__max_features": ["sqrt", "log2", None],
        },
    },
    "Random Forest": {
        "model": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__max_depth": [5, 10, 20],
            "classifier__min_samples_split": [2, 5],
            "classifier__max_features": ["sqrt", "log2"],
        },
    },
    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__learning_rate": [0.05, 0.1, 0.2],
            "classifier__max_depth": [3, 5, 7],
            "classifier__subsample": [0.8, 1.0],
        },
    },
    "XGBoost": {
        "model": XGBClassifier(
            eval_metric="logloss",
            use_label_encoder=False,
            scale_pos_weight=scale_pos,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__learning_rate": [0.05, 0.1, 0.2],
            "classifier__max_depth": [3, 5, 7],
            "classifier__subsample": [0.8, 1.0],
            "classifier__colsample_bytree": [0.8, 1.0],
        },
    },
}


# ─── EVALUATION FUNCTION ───
def evaluate_model(pipeline, X_tr, X_te, y_tr, y_te):
    """Fit on train, score on test, return metrics dict."""
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    y_proba = (
        pipeline.predict_proba(X_te)[:, 1]
        if hasattr(pipeline, "predict_proba")
        else None
    )
    metrics = {
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred, zero_division=0),
        "F1": f1_score(y_te, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_te, y_proba) if y_proba is not None else np.nan,
    }
    return metrics


# ─── MAIN PIPELINE ───
results = []

for name, config in CLASSIFIERS.items():
    print(f"\n{'─' * 60}")
    print(f"  {name}")
    print(f"{'─' * 60}")

    # ── 1. DEFAULT (no tuning) ──
    t0 = time()
    pipe_default = Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", config["model"]),
    ])

    # 5-fold CV scores (on training set)
    cv_scores = cross_val_score(
        pipe_default, X_train, y_train, cv=CV, scoring="f1"
    )

    # Train-test metrics
    metrics = evaluate_model(pipe_default, X_train, X_test, y_train, y_test)
    elapsed = time() - t0

    results.append({
        "Model": name,
        "Mode": "Default",
        "CV_Mean_F1": round(cv_scores.mean(), 4),
        "CV_Std": round(cv_scores.std(), 4),
        "Test_Accuracy": round(metrics["Accuracy"], 4),
        "Test_Precision": round(metrics["Precision"], 4),
        "Test_Recall": round(metrics["Recall"], 4),
        "Test_F1": round(metrics["F1"], 4),
        "Test_ROC_AUC": round(metrics["ROC-AUC"], 4),
        "Best_Params": "default",
        "Time_sec": round(elapsed, 2),
    })

    print(f"  [Default]  CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}  "
          f"Test_Acc={metrics['Accuracy']:.4f}  ({elapsed:.1f}s)")

    # ── 2. TUNED (RandomizedSearchCV) ──
    t0 = time()
    pipe_tuned = Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", config["model"]),
    ])

    search = RandomizedSearchCV(
        estimator=pipe_tuned,
        param_distributions=config["params"],
        n_iter=10,
        cv=CV,
        scoring="f1",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        return_train_score=False,
    )
    search.fit(X_train, y_train)

    # Evaluate best estimator on test set
    best_pipe = search.best_estimator_
    y_pred = best_pipe.predict(X_test)
    y_proba = (
        best_pipe.predict_proba(X_test)[:, 1]
        if hasattr(best_pipe, "predict_proba")
        else None
    )

    metrics_tuned = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan,
    }
    elapsed = time() - t0

    # Clean param names for display
    best_params = {
        k.replace("classifier__", ""): v
        for k, v in search.best_params_.items()
    }

    results.append({
        "Model": name,
        "Mode": "Tuned (RandomizedSearchCV)",
        "CV_Mean_F1": round(search.best_score_, 4),
        "CV_Std": round(
            search.cv_results_["std_test_score"][search.best_index_], 4
        ),
        "Test_Accuracy": round(metrics_tuned["Accuracy"], 4),
        "Test_Precision": round(metrics_tuned["Precision"], 4),
        "Test_Recall": round(metrics_tuned["Recall"], 4),
        "Test_F1": round(metrics_tuned["F1"], 4),
        "Test_ROC_AUC": round(metrics_tuned["ROC-AUC"], 4),
        "Best_Params": str(best_params),
        "Time_sec": round(elapsed, 2),
    })

    print(f"  [Tuned]    CV={search.best_score_:.4f}  "
          f"Test_Acc={metrics_tuned['Accuracy']:.4f}  ({elapsed:.1f}s)")
    print(f"  Best → {best_params}")


# ─── BUILD FINAL DATAFRAME ───
df_results = pd.DataFrame(results)
df_results = df_results.sort_values("Test_F1", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("  FINAL RESULTS — ALL CLASSIFIERS (Default + Tuned)")
print("=" * 80)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
print(df_results.to_string(index=False))

# ─── QUICK SUMMARY ───
print("\n" + "=" * 80)
print("  TOP 3 MODELS BY F1 SCORE")
print("=" * 80)
for i, row in df_results.head(3).iterrows():
    print(f"  {i+1}. {row['Model']} ({row['Mode']}) — "
          f"F1: {row['Test_F1']}  Recall: {row['Test_Recall']}  "
          f"AUC: {row['Test_ROC_AUC']}")

Note: you may need to restart the kernel to use updated packages.
Samples: 10000 | Features: 11
Train: 8000 | Test: 2000
Class distribution: {np.int64(0): np.int64(7963), np.int64(1): np.int64(2037)}
Imbalance ratio (neg/pos): 3.91

────────────────────────────────────────────────────────────
  Logistic Regression
────────────────────────────────────────────────────────────
  [Default]  CV=0.4871±0.0202  Test_Acc=0.7155  (0.4s)
  [Tuned]    CV=0.4948  Test_Acc=0.7180  (18.2s)
  Best → {'solver': 'liblinear', 'penalty': 'l1', 'C': np.float64(0.008858667904100823)}

────────────────────────────────────────────────────────────
  KNN
────────────────────────────────────────────────────────────
  [Default]  CV=0.5058±0.0134  Test_Acc=0.7245  (2.5s)
  [Tuned]    CV=0.5394  Test_Acc=0.7465  (5.7s)
  Best → {'weights': 'distance', 'p': 2, 'n_neighbors': 25, 'metric': 'euclidean'}

────────────────────────────────────────────────────────────
  SVM
───────────────────────────────────────────────

In [14]:
# Quick sanity check — preview dataframe after encoding
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain
0,1,15634602,Hargrave,619,0,42,2,0.00,1,1,1,101348.88,1,False,False
1,2,15647311,Hill,608,0,41,1,83807.86,1,0,1,112542.58,0,False,True
2,3,15619304,Onio,502,0,42,8,159660.80,3,1,0,113931.57,1,False,False
3,4,15701354,Boni,699,0,39,1,0.00,2,0,0,93826.63,0,False,False
4,5,15737888,Mitchell,850,0,43,2,125510.82,1,1,1,79084.10,0,False,True


In [15]:
# ═══════════════════════════════════════════════════════════════
# ATTEMPT 6: Added PCA dimensionality reduction
# PCA(n_components=0.95) retains 95% of variance
# Result: F1 DECREASED — PCA destroyed signal from engineered features
# Conclusion: PCA hurts when features are domain-engineered interactions,
# not redundant high-dimensional data. Rejected.
# ═══════════════════════════════════════════════════════════════
"""
=============================================================================
END-TO-END CLASSIFIER BENCHMARK PIPELINE (WITH SMOTE)
- 7 Classifiers: Logistic, KNN, SVM, Decision Tree, Random Forest,
                  Gradient Boosting, XGBoost
- Each run in DEFAULT and TUNED (RandomizedSearchCV) mode
- 5-Fold Cross-Validation on all
- SMOTE for class balancing
- Single reproducible DataFrame output
=============================================================================
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from time import time

# pip install imbalanced-learn xgboost   <-- run ONCE in terminal before running

# sklearn utilities
from sklearn.model_selection import (
    train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# SMOTE + imblearn Pipeline (applies SMOTE only on train folds, never test)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline   # NOT sklearn.pipeline.Pipeline

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# ─── REPRODUCIBILITY ───
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ─── LOAD DATA ───
df = pd.read_csv(r"C:\Users\Admin\Desktop\Notes\ML\churn\data\Churn_Modelling.csv")

# Encode Gender
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])

# One-Hot Encode Geography (FIXED — pd.get_dummies, not OHE on single column)
df = pd.get_dummies(df, columns=['Geography'], drop_first=True)

# ═══════════════════════════════════════════════════════════════════════════════
#  FEATURE ENGINEERING — this is where the real gains come from
# ═══════════════════════════════════════════════════════════════════════════════

# 1. Age² — churn vs age is non-linear (young & very old churn more)
df['Age_Squared'] = df['Age'] ** 2

# 2. Zero Balance flag — ~36% of customers have 0 balance, behave very differently
df['Balance_Is_Zero'] = (df['Balance'] == 0).astype(int)

# 3. Balance-to-Salary ratio — relative wealth matters more than absolute
df['Balance_Salary_Ratio'] = df['Balance'] / (df['EstimatedSalary'] + 1)

# 4. Age × IsActiveMember — older INACTIVE members churn far more
df['Age_x_IsActive'] = df['Age'] * df['IsActiveMember']

# 5. Age × NumOfProducts — older customers with multiple products behave differently
df['Age_x_NumProducts'] = df['Age'] * df['NumOfProducts']

# 6. Tenure / Age — loyalty relative to customer age
df['Tenure_Age_Ratio'] = df['Tenure'] / (df['Age'] + 1)

# 7. High-risk product flag — customers with 3 or 4 products churn at ~80-100%
df['Products_GT2'] = (df['NumOfProducts'] > 2).astype(int)

# 8. CreditScore bins — non-linear relationship with churn
df['CreditScore_Low'] = (df['CreditScore'] < 500).astype(int)

# 9. Senior flag — age > 45 is a known churn inflection point
df['Is_Senior'] = (df['Age'] > 45).astype(int)

# 10. Balance × NumOfProducts — high balance + many products = different risk
df['Balance_x_NumProducts'] = df['Balance'] * df['NumOfProducts']

print(f"Engineered features added. Total features: {df.shape[1] - 4}")  # minus ID cols + target

# Features and target
X = df.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Exited'])
y = df['Exited']

# DO NOT scale here — scaling happens INSIDE the pipeline (prevents data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Samples: {X.shape[0]} | Features: {X.shape[1]}")
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
class_counts = dict(zip(*np.unique(y, return_counts=True)))
print(f"Class distribution: {class_counts}")

# Compute imbalance ratio for XGBoost
neg_count = sum(1 for val in y_train if val == 0)
pos_count = sum(1 for val in y_train if val == 1)
scale_pos = neg_count / pos_count if pos_count > 0 else 1.0
print(f"Imbalance ratio (neg/pos): {scale_pos:.2f}")
print("=" * 80)

# ─── DEFINE CLASSIFIERS + HYPERPARAMETER GRIDS ───
CLASSIFIERS = {
    "Logistic Regression": {
        "model": LogisticRegression(
            max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "params": {
            "classifier__C": np.logspace(-3, 3, 20),
            "classifier__penalty": ["l1", "l2"],
            "classifier__solver": ["liblinear", "saga"],
        },
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            "classifier__n_neighbors": list(range(3, 31, 2)),
            "classifier__weights": ["uniform", "distance"],
            "classifier__metric": ["euclidean", "manhattan"],
            "classifier__p": [1, 2],
        },
    },
    "SVM": {
        "model": CalibratedClassifierCV(
            LinearSVC(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE), cv=3
        ),
        "params": {
            "classifier__estimator__C": np.logspace(-2, 2, 8),
        },
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "params": {
            "classifier__max_depth": [3, 5, 7, 10, 15, 20, None],
            "classifier__min_samples_split": [2, 5, 10, 20],
            "classifier__min_samples_leaf": [1, 2, 5, 10],
            "classifier__criterion": ["gini", "entropy"],
            "classifier__max_features": ["sqrt", "log2", None],
        },
    },
    "Random Forest": {
        "model": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__max_depth": [5, 10, 20],
            "classifier__min_samples_split": [2, 5],
            "classifier__max_features": ["sqrt", "log2"],
        },
    },
    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__learning_rate": [0.05, 0.1, 0.2],
            "classifier__max_depth": [3, 5, 7],
            "classifier__subsample": [0.8, 1.0],
        },
    },
    "XGBoost": {
        "model": XGBClassifier(
            eval_metric="logloss",
            use_label_encoder=False,
            scale_pos_weight=scale_pos,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__learning_rate": [0.05, 0.1, 0.2],
            "classifier__max_depth": [3, 5, 7],
            "classifier__subsample": [0.8, 1.0],
            "classifier__colsample_bytree": [0.8, 1.0],
        },
    },
}


# ─── EVALUATION FUNCTION ───
def evaluate_model(pipeline, X_tr, X_te, y_tr, y_te):
    """Fit on train, score on test, return metrics dict."""
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    y_proba = (
        pipeline.predict_proba(X_te)[:, 1]
        if hasattr(pipeline, "predict_proba")
        else None
    )
    metrics = {
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred, zero_division=0),
        "F1": f1_score(y_te, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_te, y_proba) if y_proba is not None else np.nan,
    }
    return metrics


# ─── MAIN PIPELINE ───
results = []

for name, config in CLASSIFIERS.items():
    print(f"\n{'─' * 60}")
    print(f"  {name}")
    print(f"{'─' * 60}")

    # ── 1. DEFAULT (no tuning) ──
    t0 = time()
    pipe_default = Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", config["model"]),
    ])

    # 5-fold CV scores (on training set)
    cv_scores = cross_val_score(
        pipe_default, X_train, y_train, cv=CV, scoring="f1"
    )

    # Train-test metrics
    metrics = evaluate_model(pipe_default, X_train, X_test, y_train, y_test)
    elapsed = time() - t0

    results.append({
        "Model": name,
        "Mode": "Default",
        "CV_Mean_F1": round(cv_scores.mean(), 4),
        "CV_Std": round(cv_scores.std(), 4),
        "Test_Accuracy": round(metrics["Accuracy"], 4),
        "Test_Precision": round(metrics["Precision"], 4),
        "Test_Recall": round(metrics["Recall"], 4),
        "Test_F1": round(metrics["F1"], 4),
        "Test_ROC_AUC": round(metrics["ROC-AUC"], 4),
        "Best_Params": "default",
        "Time_sec": round(elapsed, 2),
    })

    print(f"  [Default]  CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}  "
          f"Test_Acc={metrics['Accuracy']:.4f}  ({elapsed:.1f}s)")

    # ── 2. TUNED (RandomizedSearchCV) ──
    t0 = time()
    pipe_tuned = Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", config["model"]),
    ])

    search = RandomizedSearchCV(
        estimator=pipe_tuned,
        param_distributions=config["params"],
        n_iter=10,
        cv=CV,
        scoring="f1",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        return_train_score=False,
    )
    search.fit(X_train, y_train)

    # Evaluate best estimator on test set
    best_pipe = search.best_estimator_
    y_pred = best_pipe.predict(X_test)
    y_proba = (
        best_pipe.predict_proba(X_test)[:, 1]
        if hasattr(best_pipe, "predict_proba")
        else None
    )

    metrics_tuned = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan,
    }
    elapsed = time() - t0

    # Clean param names for display
    best_params = {
        k.replace("classifier__", ""): v
        for k, v in search.best_params_.items()
    }

    results.append({
        "Model": name,
        "Mode": "Tuned (RandomizedSearchCV)",
        "CV_Mean_F1": round(search.best_score_, 4),
        "CV_Std": round(
            search.cv_results_["std_test_score"][search.best_index_], 4
        ),
        "Test_Accuracy": round(metrics_tuned["Accuracy"], 4),
        "Test_Precision": round(metrics_tuned["Precision"], 4),
        "Test_Recall": round(metrics_tuned["Recall"], 4),
        "Test_F1": round(metrics_tuned["F1"], 4),
        "Test_ROC_AUC": round(metrics_tuned["ROC-AUC"], 4),
        "Best_Params": str(best_params),
        "Time_sec": round(elapsed, 2),
    })

    print(f"  [Tuned]    CV={search.best_score_:.4f}  "
          f"Test_Acc={metrics_tuned['Accuracy']:.4f}  ({elapsed:.1f}s)")
    print(f"  Best → {best_params}")


# ─── BUILD FINAL DATAFRAME ───
df_results = pd.DataFrame(results)
df_results = df_results.sort_values("Test_F1", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("  FINAL RESULTS — ALL CLASSIFIERS (Default + Tuned)")
print("=" * 80)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
print(df_results.to_string(index=False))

# ─── QUICK SUMMARY ───
print("\n" + "=" * 80)
print("  TOP 3 MODELS BY F1 SCORE")
print("=" * 80)
for i, row in df_results.head(3).iterrows():
    print(f"  {i+1}. {row['Model']} ({row['Mode']}) — "
          f"F1: {row['Test_F1']}  Recall: {row['Test_Recall']}  "
          f"AUC: {row['Test_ROC_AUC']}")

Engineered features added. Total features: 21
Samples: 10000 | Features: 21
Train: 8000 | Test: 2000
Class distribution: {np.int64(0): np.int64(7963), np.int64(1): np.int64(2037)}
Imbalance ratio (neg/pos): 3.91

────────────────────────────────────────────────────────────
  Logistic Regression
────────────────────────────────────────────────────────────
  [Default]  CV=0.5869±0.0127  Test_Acc=0.7830  (7.7s)
  [Tuned]    CV=0.5893  Test_Acc=0.7830  (28.9s)
  Best → {'solver': 'liblinear', 'penalty': 'l1', 'C': np.float64(0.1623776739188721)}

────────────────────────────────────────────────────────────
  KNN
────────────────────────────────────────────────────────────
  [Default]  CV=0.5321±0.0160  Test_Acc=0.7320  (1.0s)
  [Tuned]    CV=0.5605  Test_Acc=0.7435  (3.6s)
  Best → {'weights': 'distance', 'p': 2, 'n_neighbors': 25, 'metric': 'euclidean'}

────────────────────────────────────────────────────────────
  SVM
────────────────────────────────────────────────────────────
  [Defau

In [16]:
# ═══════════════════════════════════════════════════════════════
# ATTEMPT 7: Same as above but kept for iteration record
# ═══════════════════════════════════════════════════════════════
"""
=============================================================================
END-TO-END CLASSIFIER BENCHMARK PIPELINE (WITH SMOTE)
- 7 Classifiers: Logistic, KNN, SVM, Decision Tree, Random Forest,
                  Gradient Boosting, XGBoost
- Each run in DEFAULT and TUNED (RandomizedSearchCV) mode
- 5-Fold Cross-Validation on all
- SMOTE for class balancing
- Single reproducible DataFrame output
=============================================================================
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from time import time

# pip install imbalanced-learn xgboost   <-- run ONCE in terminal before running

# sklearn utilities
from sklearn.model_selection import (
    train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# SMOTE + imblearn Pipeline (applies SMOTE only on train folds, never test)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline   # NOT sklearn.pipeline.Pipeline

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# ─── REPRODUCIBILITY ───
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ─── LOAD DATA ───
df = pd.read_csv(r"C:\Users\Admin\Desktop\Notes\ML\churn\data\Churn_Modelling.csv")

# Encode Gender
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])

# One-Hot Encode Geography (FIXED — pd.get_dummies, not OHE on single column)
df = pd.get_dummies(df, columns=['Geography'], drop_first=True)

# ═══════════════════════════════════════════════════════════════════════════════
#  FEATURE ENGINEERING — this is where the real gains come from
# ═══════════════════════════════════════════════════════════════════════════════

# 1. Age² — churn vs age is non-linear (young & very old churn more)
df['Age_Squared'] = df['Age'] ** 2

# 2. Zero Balance flag — ~36% of customers have 0 balance, behave very differently
df['Balance_Is_Zero'] = (df['Balance'] == 0).astype(int)

# 3. Balance-to-Salary ratio — relative wealth matters more than absolute
df['Balance_Salary_Ratio'] = df['Balance'] / (df['EstimatedSalary'] + 1)

# 4. Age × IsActiveMember — older INACTIVE members churn far more
df['Age_x_IsActive'] = df['Age'] * df['IsActiveMember']

# 5. Age × NumOfProducts — older customers with multiple products behave differently
df['Age_x_NumProducts'] = df['Age'] * df['NumOfProducts']

# 6. Tenure / Age — loyalty relative to customer age
df['Tenure_Age_Ratio'] = df['Tenure'] / (df['Age'] + 1)

# 7. High-risk product flag — customers with 3 or 4 products churn at ~80-100%
df['Products_GT2'] = (df['NumOfProducts'] > 2).astype(int)

# 8. CreditScore bins — non-linear relationship with churn
df['CreditScore_Low'] = (df['CreditScore'] < 500).astype(int)

# 9. Senior flag — age > 45 is a known churn inflection point
df['Is_Senior'] = (df['Age'] > 45).astype(int)

# 10. Balance × NumOfProducts — high balance + many products = different risk
df['Balance_x_NumProducts'] = df['Balance'] * df['NumOfProducts']

print(f"Engineered features added. Total features: {df.shape[1] - 4}")  # minus ID cols + target

# Features and target
X = df.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Exited'])
y = df['Exited']

# DO NOT scale here — scaling happens INSIDE the pipeline (prevents data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Samples: {X.shape[0]} | Features: {X.shape[1]}")
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
class_counts = dict(zip(*np.unique(y, return_counts=True)))
print(f"Class distribution: {class_counts}")

# Compute imbalance ratio for XGBoost
neg_count = sum(1 for val in y_train if val == 0)
pos_count = sum(1 for val in y_train if val == 1)
scale_pos = neg_count / pos_count if pos_count > 0 else 1.0
print(f"Imbalance ratio (neg/pos): {scale_pos:.2f}")
print("=" * 80)

# ─── DEFINE CLASSIFIERS + HYPERPARAMETER GRIDS ───
CLASSIFIERS = {
    "Logistic Regression": {
        "model": LogisticRegression(
            max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "params": {
            "classifier__C": np.logspace(-3, 3, 20),
            "classifier__penalty": ["l1", "l2"],
            "classifier__solver": ["liblinear", "saga"],
        },
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            "classifier__n_neighbors": list(range(3, 31, 2)),
            "classifier__weights": ["uniform", "distance"],
            "classifier__metric": ["euclidean", "manhattan"],
            "classifier__p": [1, 2],
        },
    },
    "SVM": {
        "model": CalibratedClassifierCV(
            LinearSVC(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE), cv=3
        ),
        "params": {
            "classifier__estimator__C": np.logspace(-2, 2, 8),
        },
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "params": {
            "classifier__max_depth": [3, 5, 7, 10, 15, 20, None],
            "classifier__min_samples_split": [2, 5, 10, 20],
            "classifier__min_samples_leaf": [1, 2, 5, 10],
            "classifier__criterion": ["gini", "entropy"],
            "classifier__max_features": ["sqrt", "log2", None],
        },
    },
    "Random Forest": {
        "model": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__max_depth": [5, 10, 20],
            "classifier__min_samples_split": [2, 5],
            "classifier__max_features": ["sqrt", "log2"],
        },
    },
    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__learning_rate": [0.05, 0.1, 0.2],
            "classifier__max_depth": [3, 5, 7],
            "classifier__subsample": [0.8, 1.0],
        },
    },
    "XGBoost": {
        "model": XGBClassifier(
            eval_metric="logloss",
            use_label_encoder=False,
            scale_pos_weight=scale_pos,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__learning_rate": [0.05, 0.1, 0.2],
            "classifier__max_depth": [3, 5, 7],
            "classifier__subsample": [0.8, 1.0],
            "classifier__colsample_bytree": [0.8, 1.0],
        },
    },
}


# ─── EVALUATION FUNCTION ───
def evaluate_model(pipeline, X_tr, X_te, y_tr, y_te):
    """Fit on train, score on test, return metrics dict."""
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    y_proba = (
        pipeline.predict_proba(X_te)[:, 1]
        if hasattr(pipeline, "predict_proba")
        else None
    )
    metrics = {
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred, zero_division=0),
        "F1": f1_score(y_te, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_te, y_proba) if y_proba is not None else np.nan,
    }
    return metrics


# ─── MAIN PIPELINE ───
results = []

for name, config in CLASSIFIERS.items():
    print(f"\n{'─' * 60}")
    print(f"  {name}")
    print(f"{'─' * 60}")

    # ── 1. DEFAULT (no tuning) ──
    t0 = time()
    pipe_default = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE)),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", config["model"]),
    ])

    # 5-fold CV scores (on training set)
    cv_scores = cross_val_score(
        pipe_default, X_train, y_train, cv=CV, scoring="f1"
    )

    # Train-test metrics
    metrics = evaluate_model(pipe_default, X_train, X_test, y_train, y_test)
    elapsed = time() - t0

    results.append({
        "Model": name,
        "Mode": "Default",
        "CV_Mean_F1": round(cv_scores.mean(), 4),
        "CV_Std": round(cv_scores.std(), 4),
        "Test_Accuracy": round(metrics["Accuracy"], 4),
        "Test_Precision": round(metrics["Precision"], 4),
        "Test_Recall": round(metrics["Recall"], 4),
        "Test_F1": round(metrics["F1"], 4),
        "Test_ROC_AUC": round(metrics["ROC-AUC"], 4),
        "Best_Params": "default",
        "Time_sec": round(elapsed, 2),
    })

    print(f"  [Default]  CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}  "
          f"Test_Acc={metrics['Accuracy']:.4f}  ({elapsed:.1f}s)")

    # ── 2. TUNED (RandomizedSearchCV) ──
    t0 = time()
    pipe_tuned = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE)),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", config["model"]),
    ])

    search = RandomizedSearchCV(
        estimator=pipe_tuned,
        param_distributions=config["params"],
        n_iter=10,
        cv=CV,
        scoring="f1",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        return_train_score=False,
    )
    search.fit(X_train, y_train)

    # Evaluate best estimator on test set
    best_pipe = search.best_estimator_
    y_pred = best_pipe.predict(X_test)
    y_proba = (
        best_pipe.predict_proba(X_test)[:, 1]
        if hasattr(best_pipe, "predict_proba")
        else None
    )

    metrics_tuned = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan,
    }
    elapsed = time() - t0

    # Clean param names for display
    best_params = {
        k.replace("classifier__", ""): v
        for k, v in search.best_params_.items()
    }

    results.append({
        "Model": name,
        "Mode": "Tuned (RandomizedSearchCV)",
        "CV_Mean_F1": round(search.best_score_, 4),
        "CV_Std": round(
            search.cv_results_["std_test_score"][search.best_index_], 4
        ),
        "Test_Accuracy": round(metrics_tuned["Accuracy"], 4),
        "Test_Precision": round(metrics_tuned["Precision"], 4),
        "Test_Recall": round(metrics_tuned["Recall"], 4),
        "Test_F1": round(metrics_tuned["F1"], 4),
        "Test_ROC_AUC": round(metrics_tuned["ROC-AUC"], 4),
        "Best_Params": str(best_params),
        "Time_sec": round(elapsed, 2),
    })

    print(f"  [Tuned]    CV={search.best_score_:.4f}  "
          f"Test_Acc={metrics_tuned['Accuracy']:.4f}  ({elapsed:.1f}s)")
    print(f"  Best → {best_params}")


# ─── BUILD FINAL DATAFRAME ───
df_results = pd.DataFrame(results)
df_results = df_results.sort_values("Test_F1", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("  FINAL RESULTS — ALL CLASSIFIERS (Default + Tuned)")
print("=" * 80)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
print(df_results.to_string(index=False))

# ─── QUICK SUMMARY ───
print("\n" + "=" * 80)
print("  TOP 3 MODELS BY F1 SCORE")
print("=" * 80)
for i, row in df_results.head(3).iterrows():
    print(f"  {i+1}. {row['Model']} ({row['Mode']}) — "
          f"F1: {row['Test_F1']}  Recall: {row['Test_Recall']}  "
          f"AUC: {row['Test_ROC_AUC']}")

Engineered features added. Total features: 21
Samples: 10000 | Features: 21
Train: 8000 | Test: 2000
Class distribution: {np.int64(0): np.int64(7963), np.int64(1): np.int64(2037)}
Imbalance ratio (neg/pos): 3.91

────────────────────────────────────────────────────────────
  Logistic Regression
────────────────────────────────────────────────────────────
  [Default]  CV=0.5497±0.0186  Test_Acc=0.7700  (0.8s)
  [Tuned]    CV=0.5504  Test_Acc=0.7690  (1.8s)
  Best → {'solver': 'liblinear', 'penalty': 'l1', 'C': np.float64(0.1623776739188721)}

────────────────────────────────────────────────────────────
  KNN
────────────────────────────────────────────────────────────
  [Default]  CV=0.5293±0.0249  Test_Acc=0.7385  (2.0s)
  [Tuned]    CV=0.5622  Test_Acc=0.7480  (6.0s)
  Best → {'weights': 'distance', 'p': 2, 'n_neighbors': 25, 'metric': 'euclidean'}

────────────────────────────────────────────────────────────
  SVM
────────────────────────────────────────────────────────────
  [Defaul

In [17]:
# ═══════════════════════════════════════════════════════════════
# FINAL PIPELINE — 5 PHASES
# Phase 1: Feature Engineering (10 domain features)
# Phase 2: RFECV Feature Selection (21→17 features)
# Phase 3: 7 Classifiers × Default + Tuned × 5-Fold CV
# Phase 4: Ensembles (Weighted Voting + Stacking)
# Phase 5: Threshold Tuning (0.20→0.60 sweep)
#
# RESULT: GBM Default @ threshold 0.56 → F1: 0.6401
# Journey: F1 0.46 → 0.58 → 0.63 → 0.6401
# ═══════════════════════════════════════════════════════════════
"""
=============================================================================
END-TO-END CLASSIFIER PIPELINE
  Phase 1: Feature Engineering (10 domain features)
  Phase 2: Feature Selection (RFECV — finds optimal subset automatically)
  Phase 3: Classifier Benchmark (7 models × default + tuned × 5-fold CV)
=============================================================================
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from time import time

# sklearn
from sklearn.model_selection import (
    train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import RFECV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# SMOTE
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# ─── REPRODUCIBILITY ───
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 1: LOAD + FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════════════════════════
print("=" * 80)
print("  PHASE 1: FEATURE ENGINEERING")
print("=" * 80)

df = pd.read_csv(r"C:\Users\Admin\Desktop\Notes\ML\churn\data\Churn_Modelling.csv")

# Encode
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
df = pd.get_dummies(df, columns=['Geography'], drop_first=True)

# --- Engineered features ---
df['Age_Squared'] = df['Age'] ** 2
df['Balance_Is_Zero'] = (df['Balance'] == 0).astype(int)
df['Balance_Salary_Ratio'] = df['Balance'] / (df['EstimatedSalary'] + 1)
df['Age_x_IsActive'] = df['Age'] * df['IsActiveMember']
df['Age_x_NumProducts'] = df['Age'] * df['NumOfProducts']
df['Tenure_Age_Ratio'] = df['Tenure'] / (df['Age'] + 1)
df['Products_GT2'] = (df['NumOfProducts'] > 2).astype(int)
df['CreditScore_Low'] = (df['CreditScore'] < 500).astype(int)
df['Is_Senior'] = (df['Age'] > 45).astype(int)
df['Balance_x_NumProducts'] = df['Balance'] * df['NumOfProducts']

X = df.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Exited'])
y = df['Exited']
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Total features after engineering: {len(feature_names)}")
print(f"Features: {feature_names}")
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

neg_count = sum(1 for val in y_train if val == 0)
pos_count = sum(1 for val in y_train if val == 1)
scale_pos = neg_count / pos_count if pos_count > 0 else 1.0
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"Imbalance ratio: {scale_pos:.2f}")

# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 2: FEATURE SELECTION (RFECV)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("  PHASE 2: FEATURE SELECTION (RFECV with Gradient Boosting)")
print("=" * 80)

# Scale for RFECV (needed for fair feature importance comparison)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE on scaled training data for RFECV
smote = SMOTE(random_state=RANDOM_STATE)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# Use GBM (best performer) as the estimator for RFECV
gbm_selector = GradientBoostingClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.1,
    random_state=RANDOM_STATE
)

print("Running RFECV (this takes 1-2 minutes)...")
t0 = time()
rfecv = RFECV(
    estimator=gbm_selector,
    step=1,
    cv=CV,
    scoring="f1",
    min_features_to_select=5,
    n_jobs=-1,
)
rfecv.fit(X_train_resampled, y_train_resampled)
print(f"Done in {time() - t0:.1f}s")

# Results
selected_mask = rfecv.support_
selected_features = [f for f, s in zip(feature_names, selected_mask) if s]
dropped_features = [f for f, s in zip(feature_names, selected_mask) if not s]

print(f"\nOptimal number of features: {rfecv.n_features_}")
print(f"Selected ({len(selected_features)}): {selected_features}")
print(f"Dropped  ({len(dropped_features)}): {dropped_features}")

# Feature importance ranking
ranking = rfecv.ranking_
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Rank": ranking,
    "Selected": selected_mask
}).sort_values("Rank")
print("\nFeature Ranking:")
print(importance_df.to_string(index=False))

# Apply selection
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 3: CLASSIFIER BENCHMARK ON OPTIMAL FEATURES
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print(f"  PHASE 3: CLASSIFIER BENCHMARK ({len(selected_features)} selected features)")
print("=" * 80)

CLASSIFIERS = {
    "Logistic Regression": {
        "model": LogisticRegression(
            max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "params": {
            "classifier__C": np.logspace(-3, 3, 20),
            "classifier__penalty": ["l1", "l2"],
            "classifier__solver": ["liblinear", "saga"],
        },
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            "classifier__n_neighbors": list(range(3, 31, 2)),
            "classifier__weights": ["uniform", "distance"],
            "classifier__metric": ["euclidean", "manhattan"],
            "classifier__p": [1, 2],
        },
    },
    "SVM": {
        "model": CalibratedClassifierCV(
            LinearSVC(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE), cv=3
        ),
        "params": {
            "classifier__estimator__C": np.logspace(-2, 2, 8),
        },
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "params": {
            "classifier__max_depth": [3, 5, 7, 10, 15, 20, None],
            "classifier__min_samples_split": [2, 5, 10, 20],
            "classifier__min_samples_leaf": [1, 2, 5, 10],
            "classifier__criterion": ["gini", "entropy"],
            "classifier__max_features": ["sqrt", "log2", None],
        },
    },
    "Random Forest": {
        "model": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__max_depth": [5, 10, 20],
            "classifier__min_samples_split": [2, 5],
            "classifier__max_features": ["sqrt", "log2"],
        },
    },
    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__learning_rate": [0.05, 0.1, 0.2],
            "classifier__max_depth": [3, 5, 7],
            "classifier__subsample": [0.8, 1.0],
        },
    },
    "XGBoost": {
        "model": XGBClassifier(
            eval_metric="logloss",
            use_label_encoder=False,
            scale_pos_weight=scale_pos,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__learning_rate": [0.05, 0.1, 0.2],
            "classifier__max_depth": [3, 5, 7],
            "classifier__subsample": [0.8, 1.0],
            "classifier__colsample_bytree": [0.8, 1.0],
        },
    },
}


def evaluate_model(pipeline, X_tr, X_te, y_tr, y_te):
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    y_proba = (
        pipeline.predict_proba(X_te)[:, 1]
        if hasattr(pipeline, "predict_proba")
        else None
    )
    return {
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred, zero_division=0),
        "F1": f1_score(y_te, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_te, y_proba) if y_proba is not None else np.nan,
    }


results = []

for name, config in CLASSIFIERS.items():
    print(f"\n{'─' * 60}")
    print(f"  {name}")
    print(f"{'─' * 60}")

    # ── DEFAULT ──
    t0 = time()
    pipe_default = Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", config["model"]),
    ])

    cv_scores = cross_val_score(
        pipe_default, X_train_selected, y_train, cv=CV, scoring="f1"
    )
    metrics = evaluate_model(pipe_default, X_train_selected, X_test_selected, y_train, y_test)
    elapsed = time() - t0

    results.append({
        "Model": name,
        "Mode": "Default",
        "CV_Mean_F1": round(cv_scores.mean(), 4),
        "CV_Std": round(cv_scores.std(), 4),
        "Test_Accuracy": round(metrics["Accuracy"], 4),
        "Test_Precision": round(metrics["Precision"], 4),
        "Test_Recall": round(metrics["Recall"], 4),
        "Test_F1": round(metrics["F1"], 4),
        "Test_ROC_AUC": round(metrics["ROC-AUC"], 4),
        "Best_Params": "default",
        "Time_sec": round(elapsed, 2),
    })

    print(f"  [Default]  CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}  "
          f"Test_F1={metrics['F1']:.4f}  Recall={metrics['Recall']:.4f}  ({elapsed:.1f}s)")

    # ── TUNED ──
    t0 = time()
    pipe_tuned = Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", config["model"]),
    ])

    search = RandomizedSearchCV(
        estimator=pipe_tuned,
        param_distributions=config["params"],
        n_iter=10,
        cv=CV,
        scoring="f1",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        return_train_score=False,
    )
    search.fit(X_train_selected, y_train)

    best_pipe = search.best_estimator_
    y_pred = best_pipe.predict(X_test_selected)
    y_proba = (
        best_pipe.predict_proba(X_test_selected)[:, 1]
        if hasattr(best_pipe, "predict_proba")
        else None
    )

    metrics_tuned = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan,
    }
    elapsed = time() - t0

    best_params = {
        k.replace("classifier__", ""): v
        for k, v in search.best_params_.items()
    }

    results.append({
        "Model": name,
        "Mode": "Tuned (RandomizedSearchCV)",
        "CV_Mean_F1": round(search.best_score_, 4),
        "CV_Std": round(search.cv_results_["std_test_score"][search.best_index_], 4),
        "Test_Accuracy": round(metrics_tuned["Accuracy"], 4),
        "Test_Precision": round(metrics_tuned["Precision"], 4),
        "Test_Recall": round(metrics_tuned["Recall"], 4),
        "Test_F1": round(metrics_tuned["F1"], 4),
        "Test_ROC_AUC": round(metrics_tuned["ROC-AUC"], 4),
        "Best_Params": str(best_params),
        "Time_sec": round(elapsed, 2),
    })

    print(f"  [Tuned]    CV={search.best_score_:.4f}  "
          f"Test_F1={metrics_tuned['F1']:.4f}  Recall={metrics_tuned['Recall']:.4f}  ({elapsed:.1f}s)")
    print(f"  Best → {best_params}")


# ─── FINAL RESULTS ───
df_results = pd.DataFrame(results)
df_results = df_results.sort_values("Test_F1", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("  FINAL RESULTS — ALL CLASSIFIERS (Default + Tuned)")
print("=" * 80)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
print(df_results.to_string(index=False))

print("\n" + "=" * 80)
print("  TOP 3 MODELS BY F1 SCORE")
print("=" * 80)
for i, row in df_results.head(3).iterrows():
    print(f"  {i+1}. {row['Model']} ({row['Mode']}) — "
          f"F1: {row['Test_F1']}  Recall: {row['Test_Recall']}  "
          f"Precision: {row['Test_Precision']}  AUC: {row['Test_ROC_AUC']}")

print(f"\nFeatures used ({len(selected_features)}): {selected_features}")

  PHASE 1: FEATURE ENGINEERING
Total features after engineering: 21
Features: ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_Germany', 'Geography_Spain', 'Age_Squared', 'Balance_Is_Zero', 'Balance_Salary_Ratio', 'Age_x_IsActive', 'Age_x_NumProducts', 'Tenure_Age_Ratio', 'Products_GT2', 'CreditScore_Low', 'Is_Senior', 'Balance_x_NumProducts']
Train: 8000 | Test: 2000
Class distribution: {np.int64(0): np.int64(7963), np.int64(1): np.int64(2037)}
Imbalance ratio: 3.91

  PHASE 2: FEATURE SELECTION (RFECV with Gradient Boosting)
Running RFECV (this takes 1-2 minutes)...
Done in 208.3s

Optimal number of features: 17
Selected (17): ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_Germany', 'Age_Squared', 'Balance_Salary_Ratio', 'Age_x_IsActive', 'Age_x_NumProducts', 'Tenure_Age_Ratio', 'Products_GT2', 'Balance_x_NumProducts']

In [18]:
# ═══════════════════════════════════════════════════════════════
# ITERATION: Pipeline with updated feature engineering
# ═══════════════════════════════════════════════════════════════
"""
=============================================================================
END-TO-END CLASSIFIER PIPELINE
  Phase 1: Feature Engineering (10 domain features)
  Phase 2: Feature Selection (RFECV — finds optimal subset automatically)
  Phase 3: Classifier Benchmark (7 models × default + tuned × 5-fold CV)
=============================================================================
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from time import time

# sklearn
from sklearn.model_selection import (
    train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import RFECV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# SMOTE
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# ─── REPRODUCIBILITY ───
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 1: LOAD + FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════════════════════════
print("=" * 80)
print("  PHASE 1: FEATURE ENGINEERING")
print("=" * 80)

df = pd.read_csv(r"C:\Users\Admin\Desktop\Notes\ML\churn\data\Churn_Modelling.csv")

# Encode
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
df = pd.get_dummies(df, columns=['Geography'], drop_first=True)

# --- Engineered features ---
df['Age_Squared'] = df['Age'] ** 2
df['Balance_Is_Zero'] = (df['Balance'] == 0).astype(int)
df['Balance_Salary_Ratio'] = df['Balance'] / (df['EstimatedSalary'] + 1)
df['Age_x_IsActive'] = df['Age'] * df['IsActiveMember']
df['Age_x_NumProducts'] = df['Age'] * df['NumOfProducts']
df['Tenure_Age_Ratio'] = df['Tenure'] / (df['Age'] + 1)
df['Products_GT2'] = (df['NumOfProducts'] > 2).astype(int)
df['CreditScore_Low'] = (df['CreditScore'] < 500).astype(int)
df['Is_Senior'] = (df['Age'] > 45).astype(int)
df['Balance_x_NumProducts'] = df['Balance'] * df['NumOfProducts']

X = df.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Exited'])
y = df['Exited']
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Total features after engineering: {len(feature_names)}")
print(f"Features: {feature_names}")
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

neg_count = sum(1 for val in y_train if val == 0)
pos_count = sum(1 for val in y_train if val == 1)
scale_pos = neg_count / pos_count if pos_count > 0 else 1.0
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"Imbalance ratio: {scale_pos:.2f}")

# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 2: FEATURE SELECTION (RFECV)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("  PHASE 2: FEATURE SELECTION (RFECV with Gradient Boosting)")
print("=" * 80)

# Scale for RFECV (needed for fair feature importance comparison)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE on scaled training data for RFECV
smote = SMOTE(random_state=RANDOM_STATE)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# Use GBM (best performer) as the estimator for RFECV
gbm_selector = GradientBoostingClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.1,
    random_state=RANDOM_STATE
)

print("Running RFECV (this takes 1-2 minutes)...")
t0 = time()
rfecv = RFECV(
    estimator=gbm_selector,
    step=1,
    cv=CV,
    scoring="f1",
    min_features_to_select=5,
    n_jobs=-1,
)
rfecv.fit(X_train_resampled, y_train_resampled)
print(f"Done in {time() - t0:.1f}s")

# Results
selected_mask = rfecv.support_
selected_features = [f for f, s in zip(feature_names, selected_mask) if s]
dropped_features = [f for f, s in zip(feature_names, selected_mask) if not s]

print(f"\nOptimal number of features: {rfecv.n_features_}")
print(f"Selected ({len(selected_features)}): {selected_features}")
print(f"Dropped  ({len(dropped_features)}): {dropped_features}")

# Feature importance ranking
ranking = rfecv.ranking_
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Rank": ranking,
    "Selected": selected_mask
}).sort_values("Rank")
print("\nFeature Ranking:")
print(importance_df.to_string(index=False))

# Apply selection
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 3: CLASSIFIER BENCHMARK ON OPTIMAL FEATURES
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print(f"  PHASE 3: CLASSIFIER BENCHMARK ({len(selected_features)} selected features)")
print("=" * 80)

CLASSIFIERS = {
    "Logistic Regression": {
        "model": LogisticRegression(
            max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "params": {
            "classifier__C": np.logspace(-3, 3, 20),
            "classifier__penalty": ["l1", "l2"],
            "classifier__solver": ["liblinear", "saga"],
        },
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            "classifier__n_neighbors": list(range(3, 31, 2)),
            "classifier__weights": ["uniform", "distance"],
            "classifier__metric": ["euclidean", "manhattan"],
            "classifier__p": [1, 2],
        },
    },
    "SVM": {
        "model": CalibratedClassifierCV(
            LinearSVC(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE), cv=3
        ),
        "params": {
            "classifier__estimator__C": np.logspace(-2, 2, 8),
        },
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "params": {
            "classifier__max_depth": [3, 5, 7, 10, 15, 20, None],
            "classifier__min_samples_split": [2, 5, 10, 20],
            "classifier__min_samples_leaf": [1, 2, 5, 10],
            "classifier__criterion": ["gini", "entropy"],
            "classifier__max_features": ["sqrt", "log2", None],
        },
    },
    "Random Forest": {
        "model": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__max_depth": [5, 10, 20],
            "classifier__min_samples_split": [2, 5],
            "classifier__max_features": ["sqrt", "log2"],
        },
    },
    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__learning_rate": [0.05, 0.1, 0.2],
            "classifier__max_depth": [3, 5, 7],
            "classifier__subsample": [0.8, 1.0],
        },
    },
    "XGBoost": {
        "model": XGBClassifier(
            eval_metric="logloss",
            use_label_encoder=False,
            scale_pos_weight=scale_pos,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__learning_rate": [0.05, 0.1, 0.2],
            "classifier__max_depth": [3, 5, 7],
            "classifier__subsample": [0.8, 1.0],
            "classifier__colsample_bytree": [0.8, 1.0],
        },
    },
}


def evaluate_model(pipeline, X_tr, X_te, y_tr, y_te):
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    y_proba = (
        pipeline.predict_proba(X_te)[:, 1]
        if hasattr(pipeline, "predict_proba")
        else None
    )
    return {
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred, zero_division=0),
        "F1": f1_score(y_te, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_te, y_proba) if y_proba is not None else np.nan,
    }


results = []

for name, config in CLASSIFIERS.items():
    print(f"\n{'─' * 60}")
    print(f"  {name}")
    print(f"{'─' * 60}")

    # ── DEFAULT ──
    t0 = time()
    pipe_default = Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", config["model"]),
    ])

    cv_scores = cross_val_score(
        pipe_default, X_train_selected, y_train, cv=CV, scoring="f1"
    )
    metrics = evaluate_model(pipe_default, X_train_selected, X_test_selected, y_train, y_test)
    elapsed = time() - t0

    results.append({
        "Model": name,
        "Mode": "Default",
        "CV_Mean_F1": round(cv_scores.mean(), 4),
        "CV_Std": round(cv_scores.std(), 4),
        "Test_Accuracy": round(metrics["Accuracy"], 4),
        "Test_Precision": round(metrics["Precision"], 4),
        "Test_Recall": round(metrics["Recall"], 4),
        "Test_F1": round(metrics["F1"], 4),
        "Test_ROC_AUC": round(metrics["ROC-AUC"], 4),
        "Best_Params": "default",
        "Time_sec": round(elapsed, 2),
    })

    print(f"  [Default]  CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}  "
          f"Test_F1={metrics['F1']:.4f}  Recall={metrics['Recall']:.4f}  ({elapsed:.1f}s)")

    # ── TUNED ──
    t0 = time()
    pipe_tuned = Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", config["model"]),
    ])

    search = RandomizedSearchCV(
        estimator=pipe_tuned,
        param_distributions=config["params"],
        n_iter=10,
        cv=CV,
        scoring="f1",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        return_train_score=False,
    )
    search.fit(X_train_selected, y_train)

    best_pipe = search.best_estimator_
    y_pred = best_pipe.predict(X_test_selected)
    y_proba = (
        best_pipe.predict_proba(X_test_selected)[:, 1]
        if hasattr(best_pipe, "predict_proba")
        else None
    )

    metrics_tuned = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan,
    }
    elapsed = time() - t0

    best_params = {
        k.replace("classifier__", ""): v
        for k, v in search.best_params_.items()
    }

    results.append({
        "Model": name,
        "Mode": "Tuned (RandomizedSearchCV)",
        "CV_Mean_F1": round(search.best_score_, 4),
        "CV_Std": round(search.cv_results_["std_test_score"][search.best_index_], 4),
        "Test_Accuracy": round(metrics_tuned["Accuracy"], 4),
        "Test_Precision": round(metrics_tuned["Precision"], 4),
        "Test_Recall": round(metrics_tuned["Recall"], 4),
        "Test_F1": round(metrics_tuned["F1"], 4),
        "Test_ROC_AUC": round(metrics_tuned["ROC-AUC"], 4),
        "Best_Params": str(best_params),
        "Time_sec": round(elapsed, 2),
    })

    print(f"  [Tuned]    CV={search.best_score_:.4f}  "
          f"Test_F1={metrics_tuned['F1']:.4f}  Recall={metrics_tuned['Recall']:.4f}  ({elapsed:.1f}s)")
    print(f"  Best → {best_params}")


# ─── FINAL RESULTS ───
df_results = pd.DataFrame(results)
df_results = df_results.sort_values("Test_F1", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("  FINAL RESULTS — ALL CLASSIFIERS (Default + Tuned)")
print("=" * 80)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
print(df_results.to_string(index=False))

print("\n" + "=" * 80)
print("  TOP 3 MODELS BY F1 SCORE")
print("=" * 80)
for i, row in df_results.head(3).iterrows():
    print(f"  {i+1}. {row['Model']} ({row['Mode']}) — "
          f"F1: {row['Test_F1']}  Recall: {row['Test_Recall']}  "
          f"Precision: {row['Test_Precision']}  AUC: {row['Test_ROC_AUC']}")

print(f"\nFeatures used ({len(selected_features)}): {selected_features}")


# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 4: ENSEMBLE — WEIGHTED VOTING + STACKING
# ═══════════════════════════════════════════════════════════════════════════════
from sklearn.ensemble import VotingClassifier, StackingClassifier

print("\n" + "=" * 80)
print("  PHASE 4: ENSEMBLE MODELS")
print("=" * 80)

# --- Define base models with their best configs ---
gbm_default = GradientBoostingClassifier(random_state=RANDOM_STATE)

gbm_tuned = GradientBoostingClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.05, subsample=1.0,
    random_state=RANDOM_STATE
)

rf_tuned = RandomForestClassifier(
    n_estimators=50, min_samples_split=2, max_features="sqrt", max_depth=10,
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
)

xgb_tuned = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=1.0, colsample_bytree=1.0,
    scale_pos_weight=scale_pos, eval_metric="logloss",
    use_label_encoder=False, random_state=RANDOM_STATE, n_jobs=-1
)

# --- F1 scores for weighting ---
f1_gbm_def = 0.6277
f1_gbm_tuned = 0.6270
f1_rf_tuned = 0.6065
f1_xgb_tuned = 0.5753

# --- Scale + SMOTE the data once for ensemble evaluation ---
scaler_ens = StandardScaler()
X_train_sc = scaler_ens.fit_transform(X_train_selected)
X_test_sc = scaler_ens.transform(X_test_selected)

smote_ens = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote_ens.fit_resample(X_train_sc, y_train)

# --- Evaluation helper ---
def eval_ensemble(model, X_tr, y_tr, X_te, y_te, label):
    t0 = time()
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else None

    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec = recall_score(y_te, y_pred, zero_division=0)
    f1 = f1_score(y_te, y_pred, zero_division=0)
    auc = roc_auc_score(y_te, y_proba) if y_proba is not None else np.nan
    elapsed = time() - t0

    print(f"\n  {label}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1:        {f1:.4f}")
    print(f"  ROC-AUC:   {auc:.4f}")
    print(f"  Time:      {elapsed:.1f}s")

    return {
        "Model": label, "Test_Accuracy": round(acc, 4),
        "Test_Precision": round(prec, 4), "Test_Recall": round(rec, 4),
        "Test_F1": round(f1, 4), "Test_ROC_AUC": round(auc, 4),
        "Time_sec": round(elapsed, 2),
    }


ensemble_results = []

# ── ENSEMBLE A: Your pick (GBM Default + GBM Tuned + RF Tuned) ──
print("\n" + "─" * 60)
print("  ENSEMBLE A: GBM Default + GBM Tuned + RF Tuned (your pick)")
print("─" * 60)

voting_A = VotingClassifier(
    estimators=[
        ("gbm_def", gbm_default),
        ("gbm_tuned", gbm_tuned),
        ("rf_tuned", rf_tuned),
    ],
    voting="soft",
    weights=[f1_gbm_def, f1_gbm_tuned, f1_rf_tuned],
)
r = eval_ensemble(voting_A, X_train_sm, y_train_sm, X_test_sc, y_test,
                  "Weighted Voting A (GBM+GBM+RF)")
ensemble_results.append(r)

# ── ENSEMBLE B: Diverse (GBM Default + RF Tuned + XGBoost Tuned) ──
print("\n" + "─" * 60)
print("  ENSEMBLE B: GBM Default + RF Tuned + XGBoost Tuned (diverse)")
print("─" * 60)

voting_B = VotingClassifier(
    estimators=[
        ("gbm_def", gbm_default),
        ("rf_tuned", rf_tuned),
        ("xgb_tuned", xgb_tuned),
    ],
    voting="soft",
    weights=[f1_gbm_def, f1_rf_tuned, f1_xgb_tuned],
)
r = eval_ensemble(voting_B, X_train_sm, y_train_sm, X_test_sc, y_test,
                  "Weighted Voting B (GBM+RF+XGB)")
ensemble_results.append(r)

# ── STACKING: GBM + RF + XGBoost → Logistic meta-learner ──
print("\n" + "─" * 60)
print("  STACKING: GBM + RF + XGBoost → LogisticRegression meta-learner")
print("─" * 60)

stacking = StackingClassifier(
    estimators=[
        ("gbm_def", gbm_default),
        ("rf_tuned", rf_tuned),
        ("xgb_tuned", xgb_tuned),
    ],
    final_estimator=LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
    cv=5,
    passthrough=True,   # feeds original features + base predictions to meta-learner
    n_jobs=-1,
)
r = eval_ensemble(stacking, X_train_sm, y_train_sm, X_test_sc, y_test,
                  "Stacking (GBM+RF+XGB → LR)")
ensemble_results.append(r)

# ── COMPARISON TABLE ──
# Add best individual model for reference
ensemble_results.append({
    "Model": "Best Individual (GBM Default)",
    "Test_Accuracy": 0.8375, "Test_Precision": 0.5880,
    "Test_Recall": 0.6732, "Test_F1": 0.6277, "Test_ROC_AUC": 0.8662,
    "Time_sec": 20.09,
})

df_ensemble = pd.DataFrame(ensemble_results).sort_values("Test_F1", ascending=False)

print("\n" + "=" * 80)
print("  ENSEMBLE vs INDIVIDUAL — FINAL COMPARISON")
print("=" * 80)
print(df_ensemble.to_string(index=False))

print("\n" + "=" * 80)
print("  WINNER")
print("=" * 80)
winner = df_ensemble.iloc[0]
print(f"  {winner['Model']}")
print(f"  F1: {winner['Test_F1']}  Recall: {winner['Test_Recall']}  "
      f"Precision: {winner['Test_Precision']}  AUC: {winner['Test_ROC_AUC']}")

  PHASE 1: FEATURE ENGINEERING
Total features after engineering: 21
Features: ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_Germany', 'Geography_Spain', 'Age_Squared', 'Balance_Is_Zero', 'Balance_Salary_Ratio', 'Age_x_IsActive', 'Age_x_NumProducts', 'Tenure_Age_Ratio', 'Products_GT2', 'CreditScore_Low', 'Is_Senior', 'Balance_x_NumProducts']
Train: 8000 | Test: 2000
Class distribution: {np.int64(0): np.int64(7963), np.int64(1): np.int64(2037)}
Imbalance ratio: 3.91

  PHASE 2: FEATURE SELECTION (RFECV with Gradient Boosting)
Running RFECV (this takes 1-2 minutes)...
Done in 318.3s

Optimal number of features: 17
Selected (17): ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_Germany', 'Age_Squared', 'Balance_Salary_Ratio', 'Age_x_IsActive', 'Age_x_NumProducts', 'Tenure_Age_Ratio', 'Products_GT2', 'Balance_x_NumProducts']

In [19]:
# ═══════════════════════════════════════════════════════════════
# FINAL VERSION: Complete 5-Phase Pipeline
# Phase 1: Feature Engineering → 21 features created
# Phase 2: RFECV → Dropped Geography_Spain, Balance_Is_Zero,
#           CreditScore_Low, Is_Senior (noise features)
# Phase 3: 7 classifiers benchmarked → GBM Default wins (F1: 0.6277)
# Phase 4: Ensembles → Voting A beats individual by 0.007
# Phase 5: Threshold 0.56 → Final F1: 0.6401, Recall: 0.6314
# ═══════════════════════════════════════════════════════════════
"""
=============================================================================
END-TO-END CLASSIFIER PIPELINE
  Phase 1: Feature Engineering (10 domain features)
  Phase 2: Feature Selection (RFECV — finds optimal subset automatically)
  Phase 3: Classifier Benchmark (7 models × default + tuned × 5-fold CV)
=============================================================================
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from time import time

# sklearn
from sklearn.model_selection import (
    train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import RFECV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# SMOTE
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# ─── REPRODUCIBILITY ───
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 1: LOAD + FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════════════════════════
print("=" * 80)
print("  PHASE 1: FEATURE ENGINEERING")
print("=" * 80)

df = pd.read_csv(r"C:\Users\Admin\Desktop\Notes\ML\churn\data\Churn_Modelling.csv")

# Encode
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
df = pd.get_dummies(df, columns=['Geography'], drop_first=True)

# --- Engineered features ---
df['Age_Squared'] = df['Age'] ** 2
df['Balance_Is_Zero'] = (df['Balance'] == 0).astype(int)
df['Balance_Salary_Ratio'] = df['Balance'] / (df['EstimatedSalary'] + 1)
df['Age_x_IsActive'] = df['Age'] * df['IsActiveMember']
df['Age_x_NumProducts'] = df['Age'] * df['NumOfProducts']
df['Tenure_Age_Ratio'] = df['Tenure'] / (df['Age'] + 1)
df['Products_GT2'] = (df['NumOfProducts'] > 2).astype(int)
df['CreditScore_Low'] = (df['CreditScore'] < 500).astype(int)
df['Is_Senior'] = (df['Age'] > 45).astype(int)
df['Balance_x_NumProducts'] = df['Balance'] * df['NumOfProducts']

X = df.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Exited'])
y = df['Exited']
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Total features after engineering: {len(feature_names)}")
print(f"Features: {feature_names}")
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

neg_count = sum(1 for val in y_train if val == 0)
pos_count = sum(1 for val in y_train if val == 1)
scale_pos = neg_count / pos_count if pos_count > 0 else 1.0
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"Imbalance ratio: {scale_pos:.2f}")

# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 2: FEATURE SELECTION (RFECV)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("  PHASE 2: FEATURE SELECTION (RFECV with Gradient Boosting)")
print("=" * 80)

# Scale for RFECV (needed for fair feature importance comparison)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE on scaled training data for RFECV
smote = SMOTE(random_state=RANDOM_STATE)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# Use GBM (best performer) as the estimator for RFECV
gbm_selector = GradientBoostingClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.1,
    random_state=RANDOM_STATE
)

print("Running RFECV (this takes 1-2 minutes)...")
t0 = time()
rfecv = RFECV(
    estimator=gbm_selector,
    step=1,
    cv=CV,
    scoring="f1",
    min_features_to_select=5,
    n_jobs=-1,
)
rfecv.fit(X_train_resampled, y_train_resampled)
print(f"Done in {time() - t0:.1f}s")

# Results
selected_mask = rfecv.support_
selected_features = [f for f, s in zip(feature_names, selected_mask) if s]
dropped_features = [f for f, s in zip(feature_names, selected_mask) if not s]

print(f"\nOptimal number of features: {rfecv.n_features_}")
print(f"Selected ({len(selected_features)}): {selected_features}")
print(f"Dropped  ({len(dropped_features)}): {dropped_features}")

# Feature importance ranking
ranking = rfecv.ranking_
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Rank": ranking,
    "Selected": selected_mask
}).sort_values("Rank")
print("\nFeature Ranking:")
print(importance_df.to_string(index=False))

# Apply selection
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 3: CLASSIFIER BENCHMARK ON OPTIMAL FEATURES
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print(f"  PHASE 3: CLASSIFIER BENCHMARK ({len(selected_features)} selected features)")
print("=" * 80)

CLASSIFIERS = {
    "Logistic Regression": {
        "model": LogisticRegression(
            max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "params": {
            "classifier__C": np.logspace(-3, 3, 20),
            "classifier__penalty": ["l1", "l2"],
            "classifier__solver": ["liblinear", "saga"],
        },
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            "classifier__n_neighbors": list(range(3, 31, 2)),
            "classifier__weights": ["uniform", "distance"],
            "classifier__metric": ["euclidean", "manhattan"],
            "classifier__p": [1, 2],
        },
    },
    "SVM": {
        "model": CalibratedClassifierCV(
            LinearSVC(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE), cv=3
        ),
        "params": {
            "classifier__estimator__C": np.logspace(-2, 2, 8),
        },
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "params": {
            "classifier__max_depth": [3, 5, 7, 10, 15, 20, None],
            "classifier__min_samples_split": [2, 5, 10, 20],
            "classifier__min_samples_leaf": [1, 2, 5, 10],
            "classifier__criterion": ["gini", "entropy"],
            "classifier__max_features": ["sqrt", "log2", None],
        },
    },
    "Random Forest": {
        "model": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__max_depth": [5, 10, 20],
            "classifier__min_samples_split": [2, 5],
            "classifier__max_features": ["sqrt", "log2"],
        },
    },
    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__learning_rate": [0.05, 0.1, 0.2],
            "classifier__max_depth": [3, 5, 7],
            "classifier__subsample": [0.8, 1.0],
        },
    },
    "XGBoost": {
        "model": XGBClassifier(
            eval_metric="logloss",
            use_label_encoder=False,
            scale_pos_weight=scale_pos,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "params": {
            "classifier__n_estimators": [50, 100, 200],
            "classifier__learning_rate": [0.05, 0.1, 0.2],
            "classifier__max_depth": [3, 5, 7],
            "classifier__subsample": [0.8, 1.0],
            "classifier__colsample_bytree": [0.8, 1.0],
        },
    },
}


def evaluate_model(pipeline, X_tr, X_te, y_tr, y_te):
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    y_proba = (
        pipeline.predict_proba(X_te)[:, 1]
        if hasattr(pipeline, "predict_proba")
        else None
    )
    return {
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred, zero_division=0),
        "F1": f1_score(y_te, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_te, y_proba) if y_proba is not None else np.nan,
    }


results = []

for name, config in CLASSIFIERS.items():
    print(f"\n{'─' * 60}")
    print(f"  {name}")
    print(f"{'─' * 60}")

    # ── DEFAULT ──
    t0 = time()
    pipe_default = Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", config["model"]),
    ])

    cv_scores = cross_val_score(
        pipe_default, X_train_selected, y_train, cv=CV, scoring="f1"
    )
    metrics = evaluate_model(pipe_default, X_train_selected, X_test_selected, y_train, y_test)
    elapsed = time() - t0

    results.append({
        "Model": name,
        "Mode": "Default",
        "CV_Mean_F1": round(cv_scores.mean(), 4),
        "CV_Std": round(cv_scores.std(), 4),
        "Test_Accuracy": round(metrics["Accuracy"], 4),
        "Test_Precision": round(metrics["Precision"], 4),
        "Test_Recall": round(metrics["Recall"], 4),
        "Test_F1": round(metrics["F1"], 4),
        "Test_ROC_AUC": round(metrics["ROC-AUC"], 4),
        "Best_Params": "default",
        "Time_sec": round(elapsed, 2),
    })

    print(f"  [Default]  CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}  "
          f"Test_F1={metrics['F1']:.4f}  Recall={metrics['Recall']:.4f}  ({elapsed:.1f}s)")

    # ── TUNED ──
    t0 = time()
    pipe_tuned = Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", config["model"]),
    ])

    search = RandomizedSearchCV(
        estimator=pipe_tuned,
        param_distributions=config["params"],
        n_iter=10,
        cv=CV,
        scoring="f1",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        return_train_score=False,
    )
    search.fit(X_train_selected, y_train)

    best_pipe = search.best_estimator_
    y_pred = best_pipe.predict(X_test_selected)
    y_proba = (
        best_pipe.predict_proba(X_test_selected)[:, 1]
        if hasattr(best_pipe, "predict_proba")
        else None
    )

    metrics_tuned = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan,
    }
    elapsed = time() - t0

    best_params = {
        k.replace("classifier__", ""): v
        for k, v in search.best_params_.items()
    }

    results.append({
        "Model": name,
        "Mode": "Tuned (RandomizedSearchCV)",
        "CV_Mean_F1": round(search.best_score_, 4),
        "CV_Std": round(search.cv_results_["std_test_score"][search.best_index_], 4),
        "Test_Accuracy": round(metrics_tuned["Accuracy"], 4),
        "Test_Precision": round(metrics_tuned["Precision"], 4),
        "Test_Recall": round(metrics_tuned["Recall"], 4),
        "Test_F1": round(metrics_tuned["F1"], 4),
        "Test_ROC_AUC": round(metrics_tuned["ROC-AUC"], 4),
        "Best_Params": str(best_params),
        "Time_sec": round(elapsed, 2),
    })

    print(f"  [Tuned]    CV={search.best_score_:.4f}  "
          f"Test_F1={metrics_tuned['F1']:.4f}  Recall={metrics_tuned['Recall']:.4f}  ({elapsed:.1f}s)")
    print(f"  Best → {best_params}")


# ─── FINAL RESULTS ───
df_results = pd.DataFrame(results)
df_results = df_results.sort_values("Test_F1", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("  FINAL RESULTS — ALL CLASSIFIERS (Default + Tuned)")
print("=" * 80)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
print(df_results.to_string(index=False))

print("\n" + "=" * 80)
print("  TOP 3 MODELS BY F1 SCORE")
print("=" * 80)
for i, row in df_results.head(3).iterrows():
    print(f"  {i+1}. {row['Model']} ({row['Mode']}) — "
          f"F1: {row['Test_F1']}  Recall: {row['Test_Recall']}  "
          f"Precision: {row['Test_Precision']}  AUC: {row['Test_ROC_AUC']}")

print(f"\nFeatures used ({len(selected_features)}): {selected_features}")


# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 4: ENSEMBLE — WEIGHTED VOTING + STACKING
# ═══════════════════════════════════════════════════════════════════════════════
from sklearn.ensemble import VotingClassifier, StackingClassifier

print("\n" + "=" * 80)
print("  PHASE 4: ENSEMBLE MODELS")
print("=" * 80)

# --- Define base models with their best configs ---
gbm_default = GradientBoostingClassifier(random_state=RANDOM_STATE)

gbm_tuned = GradientBoostingClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.05, subsample=1.0,
    random_state=RANDOM_STATE
)

rf_tuned = RandomForestClassifier(
    n_estimators=50, min_samples_split=2, max_features="sqrt", max_depth=10,
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
)

xgb_tuned = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=1.0, colsample_bytree=1.0,
    scale_pos_weight=scale_pos, eval_metric="logloss",
    use_label_encoder=False, random_state=RANDOM_STATE, n_jobs=-1
)

# --- F1 scores for weighting ---
f1_gbm_def = 0.6277
f1_gbm_tuned = 0.6270
f1_rf_tuned = 0.6065
f1_xgb_tuned = 0.5753

# --- Scale + SMOTE the data once for ensemble evaluation ---
scaler_ens = StandardScaler()
X_train_sc = scaler_ens.fit_transform(X_train_selected)
X_test_sc = scaler_ens.transform(X_test_selected)

smote_ens = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote_ens.fit_resample(X_train_sc, y_train)

# --- Evaluation helper ---
def eval_ensemble(model, X_tr, y_tr, X_te, y_te, label):
    t0 = time()
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else None

    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec = recall_score(y_te, y_pred, zero_division=0)
    f1 = f1_score(y_te, y_pred, zero_division=0)
    auc = roc_auc_score(y_te, y_proba) if y_proba is not None else np.nan
    elapsed = time() - t0

    print(f"\n  {label}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1:        {f1:.4f}")
    print(f"  ROC-AUC:   {auc:.4f}")
    print(f"  Time:      {elapsed:.1f}s")

    return {
        "Model": label, "Test_Accuracy": round(acc, 4),
        "Test_Precision": round(prec, 4), "Test_Recall": round(rec, 4),
        "Test_F1": round(f1, 4), "Test_ROC_AUC": round(auc, 4),
        "Time_sec": round(elapsed, 2),
    }


ensemble_results = []

# ── ENSEMBLE A: Your pick (GBM Default + GBM Tuned + RF Tuned) ──
print("\n" + "─" * 60)
print("  ENSEMBLE A: GBM Default + GBM Tuned + RF Tuned (your pick)")
print("─" * 60)

voting_A = VotingClassifier(
    estimators=[
        ("gbm_def", gbm_default),
        ("gbm_tuned", gbm_tuned),
        ("rf_tuned", rf_tuned),
    ],
    voting="soft",
    weights=[f1_gbm_def, f1_gbm_tuned, f1_rf_tuned],
)
r = eval_ensemble(voting_A, X_train_sm, y_train_sm, X_test_sc, y_test,
                  "Weighted Voting A (GBM+GBM+RF)")
ensemble_results.append(r)

# ── ENSEMBLE B: Diverse (GBM Default + RF Tuned + XGBoost Tuned) ──
print("\n" + "─" * 60)
print("  ENSEMBLE B: GBM Default + RF Tuned + XGBoost Tuned (diverse)")
print("─" * 60)

voting_B = VotingClassifier(
    estimators=[
        ("gbm_def", gbm_default),
        ("rf_tuned", rf_tuned),
        ("xgb_tuned", xgb_tuned),
    ],
    voting="soft",
    weights=[f1_gbm_def, f1_rf_tuned, f1_xgb_tuned],
)
r = eval_ensemble(voting_B, X_train_sm, y_train_sm, X_test_sc, y_test,
                  "Weighted Voting B (GBM+RF+XGB)")
ensemble_results.append(r)

# ── STACKING: GBM + RF + XGBoost → Logistic meta-learner ──
print("\n" + "─" * 60)
print("  STACKING: GBM + RF + XGBoost → LogisticRegression meta-learner")
print("─" * 60)

stacking = StackingClassifier(
    estimators=[
        ("gbm_def", gbm_default),
        ("rf_tuned", rf_tuned),
        ("xgb_tuned", xgb_tuned),
    ],
    final_estimator=LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
    cv=5,
    passthrough=True,   # feeds original features + base predictions to meta-learner
    n_jobs=-1,
)
r = eval_ensemble(stacking, X_train_sm, y_train_sm, X_test_sc, y_test,
                  "Stacking (GBM+RF+XGB → LR)")
ensemble_results.append(r)

# ── COMPARISON TABLE ──
# Add best individual model for reference
ensemble_results.append({
    "Model": "Best Individual (GBM Default)",
    "Test_Accuracy": 0.8375, "Test_Precision": 0.5880,
    "Test_Recall": 0.6732, "Test_F1": 0.6277, "Test_ROC_AUC": 0.8662,
    "Time_sec": 20.09,
})

df_ensemble = pd.DataFrame(ensemble_results).sort_values("Test_F1", ascending=False)

print("\n" + "=" * 80)
print("  ENSEMBLE vs INDIVIDUAL — FINAL COMPARISON")
print("=" * 80)
print(df_ensemble.to_string(index=False))

print("\n" + "=" * 80)
print("  WINNER")
print("=" * 80)
winner = df_ensemble.iloc[0]
print(f"  {winner['Model']}")
print(f"  F1: {winner['Test_F1']}  Recall: {winner['Test_Recall']}  "
      f"Precision: {winner['Test_Precision']}  AUC: {winner['Test_ROC_AUC']}")


# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 5: THRESHOLD TUNING
#  Default is 0.5 — for imbalanced data, optimal is usually 0.30-0.45
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("  PHASE 5: THRESHOLD TUNING")
print("=" * 80)

# All models are already fitted from Phase 4 — just need predict_proba
models_to_tune = {
    "GBM Default": gbm_default,
    "Voting A (GBM+GBM+RF)": voting_A,
    "Voting B (GBM+RF+XGB)": voting_B,
    "Stacking (GBM+RF+XGB→LR)": stacking,
}

# Also fit standalone GBM on SMOTE data (it was fit inside pipeline earlier, need fresh fit)
gbm_standalone = GradientBoostingClassifier(random_state=RANDOM_STATE)
gbm_standalone.fit(X_train_sm, y_train_sm)
models_to_tune["GBM Default"] = gbm_standalone

thresholds = np.arange(0.20, 0.61, 0.01)
threshold_results = []

for model_name, model in models_to_tune.items():
    print(f"\n{'─' * 60}")
    print(f"  {model_name}")
    print(f"{'─' * 60}")

    y_proba = model.predict_proba(X_test_sc)[:, 1]

    best_f1 = 0
    best_thresh = 0.5
    best_metrics = {}

    # Scan all thresholds
    for thresh in thresholds:
        y_pred_t = (y_proba >= thresh).astype(int)
        f1_t = f1_score(y_test, y_pred_t, zero_division=0)
        if f1_t > best_f1:
            best_f1 = f1_t
            best_thresh = thresh
            best_metrics = {
                "Accuracy": accuracy_score(y_test, y_pred_t),
                "Precision": precision_score(y_test, y_pred_t, zero_division=0),
                "Recall": recall_score(y_test, y_pred_t, zero_division=0),
                "F1": f1_t,
                "ROC-AUC": roc_auc_score(y_test, y_proba),
            }

    # Default threshold (0.5) metrics for comparison
    y_pred_default = (y_proba >= 0.5).astype(int)
    f1_default = f1_score(y_test, y_pred_default, zero_division=0)
    rec_default = recall_score(y_test, y_pred_default, zero_division=0)
    prec_default = precision_score(y_test, y_pred_default, zero_division=0)

    improvement = best_metrics["F1"] - f1_default

    print(f"  Default (0.50):  F1={f1_default:.4f}  Prec={prec_default:.4f}  Recall={rec_default:.4f}")
    print(f"  Optimal ({best_thresh:.2f}):  F1={best_metrics['F1']:.4f}  "
          f"Prec={best_metrics['Precision']:.4f}  Recall={best_metrics['Recall']:.4f}")
    print(f"  F1 improvement:  {improvement:+.4f} ({improvement/f1_default*100:+.1f}%)")

    threshold_results.append({
        "Model": model_name,
        "Default_Threshold": 0.50,
        "Default_F1": round(f1_default, 4),
        "Default_Precision": round(prec_default, 4),
        "Default_Recall": round(rec_default, 4),
        "Optimal_Threshold": round(best_thresh, 2),
        "Tuned_F1": round(best_metrics["F1"], 4),
        "Tuned_Precision": round(best_metrics["Precision"], 4),
        "Tuned_Recall": round(best_metrics["Recall"], 4),
        "Tuned_ROC_AUC": round(best_metrics["ROC-AUC"], 4),
        "F1_Gain": round(improvement, 4),
    })

# ── FINAL COMPARISON TABLE ──
df_thresh = pd.DataFrame(threshold_results).sort_values("Tuned_F1", ascending=False)

print("\n" + "=" * 80)
print("  THRESHOLD TUNING — FULL COMPARISON")
print("=" * 80)
print(df_thresh.to_string(index=False))

# ── GRAND FINAL: BEST MODEL EVER ──
print("\n" + "=" * 80)
print("  ★ GRAND FINAL — BEST CONFIGURATION ACROSS ALL PHASES ★")
print("=" * 80)

best = df_thresh.iloc[0]
print(f"  Model:     {best['Model']}")
print(f"  Threshold: {best['Optimal_Threshold']}")
print(f"  F1:        {best['Tuned_F1']}")
print(f"  Recall:    {best['Tuned_Recall']}")
print(f"  Precision: {best['Tuned_Precision']}")
print(f"  ROC-AUC:   {best['Tuned_ROC_AUC']}")
print(f"")
print(f"  Journey:   F1 0.46 (raw) → 0.58 (SMOTE) → 0.63 (features) → {best['Tuned_F1']} (threshold)")
print(f"  Features:  {len(selected_features)} selected by RFECV")
print(f"  {selected_features}")

  PHASE 1: FEATURE ENGINEERING
Total features after engineering: 21
Features: ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_Germany', 'Geography_Spain', 'Age_Squared', 'Balance_Is_Zero', 'Balance_Salary_Ratio', 'Age_x_IsActive', 'Age_x_NumProducts', 'Tenure_Age_Ratio', 'Products_GT2', 'CreditScore_Low', 'Is_Senior', 'Balance_x_NumProducts']
Train: 8000 | Test: 2000
Class distribution: {np.int64(0): np.int64(7963), np.int64(1): np.int64(2037)}
Imbalance ratio: 3.91

  PHASE 2: FEATURE SELECTION (RFECV with Gradient Boosting)
Running RFECV (this takes 1-2 minutes)...
Done in 273.4s

Optimal number of features: 17
Selected (17): ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_Germany', 'Age_Squared', 'Balance_Salary_Ratio', 'Age_x_IsActive', 'Age_x_NumProducts', 'Tenure_Age_Ratio', 'Products_GT2', 'Balance_x_NumProducts']